In [1]:
import pandas as pd
from tqdm import tqdm
tqdm.pandas()
import numpy as np
import datetime as dt
import math as math
pd.set_option("display.max_columns", None)
from pandas.tseries.offsets import MonthEnd

In [2]:
import_folder_path = r"..\firm_calc1_output"
output_folder_path = "firm_fin_output"
supporting_folder_path = "supporting_datafiles"

### Identifiers: Prowess Company Code x NSE Symbol [Done later]

In [3]:
compCodeNse = pd.read_excel(rf"{supporting_folder_path}\Prowess Code_NSE Symbol.xlsx")
compCodeNse

,Company Name,Prowess company code,NSE symbol
0,'K' Steamship Agencies Pvt. Ltd.,3,NaN
1,'X'Clusive Business Centre Pvt. Ltd.,307865,NaN
2,1 To 1 Help.Net Pvt. Ltd.,591675,NaN
3,10C India Internet Pvt. Ltd.,556976,NaN
4,10I Commerce Services Pvt. Ltd.,560502,NaN
...,...,...,...
55307,Zylog Systems Ltd.,275793,ZYLOG
55308,Zyma Laboratories Ltd. [Merged],275794,NaN
55309,Zyphar'S Pharmaceutics Pvt. Ltd.,565342,NaN
55310,Zytel Agencies Ltd.,275795,NaN


### READING ALL DATA FILES

#### 16 Ownership Data

In [4]:
own1 = pd.read_excel(rf"{supporting_folder_path}\Prowess IQ - Ownership Data.xlsx", sheet_name = 0, header = [0,1])
own2 = pd.read_excel(rf"{supporting_folder_path}\Prowess IQ - Ownership Data.xlsx", sheet_name = 1, header = [0,1])

In [5]:
own11 = own1.drop([("Unnamed: 1_level_0","NSE symbol")], axis = 1).set_index([("Unnamed: 0_level_0","Company Name")])
own12 = own11.unstack().unstack(level = 1).reset_index()
own13 = own12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
own13["AsOnDate"] = pd.to_datetime(own13["AsOnDate"], format = "%b %Y" ) + MonthEnd(0)

own21 = own2.drop([("Unnamed: 1_level_0","NSE symbol")], axis = 1).set_index([("Unnamed: 0_level_0","Company Name")])
own22 = own21.unstack().unstack(level = 1).reset_index()
own23 = own22.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
own23["AsOnDate"] = pd.to_datetime(own23["AsOnDate"], format = "%b %Y" ) + MonthEnd(0)

In [6]:
own = pd.concat([own13, own23], axis = 0).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop= True).drop_duplicates()
own

,AsOnDate,Company Name,Total Shares (In %) - Shares held,Promoters (In %) - Shares held,Indian Promoters (In %) - Shares held,Foreign Promoters (In %) - Shares held,Non-promoters (In %) - Shares held,Non-promoter Institutions (In %) - Shares held,Non-promoter Non-institutions (In %) - Shares held
0,2005-03-31,20 Microns Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2005-06-30,20 Microns Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2005-09-30,20 Microns Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2005-12-31,20 Microns Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2006-03-31,20 Microns Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
297487,2023-03-31,Zylog Systems Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN
297488,2023-06-30,Zylog Systems Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN
297489,2023-09-30,Zylog Systems Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN
297490,2023-12-31,Zylog Systems Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### 1 Financial Data

In [7]:
fin1 = pd.read_excel(rf"{supporting_folder_path}\Prowess IQ - Financial Data.xlsx", sheet_name = 0, header = [0,1])

fin2 = pd.read_excel(rf"{supporting_folder_path}\Prowess IQ - Financial Data.xlsx", sheet_name = 1, header = [0,1])

In [8]:
# fin1 = fin_identifiers.drop(fin_identifiers.columns[[1,3,4,5,6,7,8,9,10,11,12,13]], axis = 1)

In [9]:
fin11 = fin1.drop([("Unnamed: 1_level_0","NSE symbol")], axis = 1).set_index([("Unnamed: 0_level_0","Company Name")])
fin12 = fin11.unstack().unstack(level = 1).reset_index()
fin13 = fin12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
fin13["AsOnDate"] = pd.to_datetime(fin13["AsOnDate"], format = "%b %Y" ) + MonthEnd(0)

fin21 = fin2.drop([("Unnamed: 1_level_0","NSE symbol")], axis = 1).set_index([("Unnamed: 0_level_0","Company Name")])
fin22 = fin21.unstack().unstack(level = 1).reset_index()
fin23 = fin22.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
fin23["AsOnDate"] = pd.to_datetime(fin23["AsOnDate"], format = "%b %Y" ) + MonthEnd(0)

In [10]:
fin = pd.concat([fin13, fin23], axis = 0).sort_values(by = ["Company Name", "AsOnDate"])\
.reset_index(drop= True).drop_duplicates()\
.drop(["Cash (outflow) due to purchase of fixed assets", "Cash inflow due to sale of fixed assets",
       "Cash flow due to purchase of fixed assets",
       "Net cash inflow or (outflow) from investing activities"], axis = 1)

fin

,AsOnDate,Company Name,Sales,Profit after tax,PBDITA,Return on total assets,Total assets,Operating profit of non-financial companies,Operating profit of financial companies,Debt to equity ratio (times),Long term borrowings incl current portion,Long term borrowings excl current portion,Debt,Trade payables (Old Sch. VI),Subscribed preference capital,Paid up preference capital (net of forfeited preference capital) (Old Sch. VI),Paid up preference capital (net of forfeited preference capital),Paid up preference shares,Total other receivables,Net cash flow from operating activities,Research & development expenses,Inter-corporate loans (Old Sch. VI)
0,2005-03-31,20 Microns Ltd.,578.3,-63.2,70.0,-8.78,698.5,66.3,2.3,2.08,NaN,NaN,389.1,70.6,NaN,NaN,NaN,NaN,NaN,68.2,NaN,NaN
1,2006-03-31,20 Microns Ltd.,720.8,18.0,83.1,2.58,742.9,80.0,9.4,1.98,NaN,NaN,399.6,57.7,NaN,NaN,NaN,NaN,0.9,53.8,1.9,NaN
2,2007-03-31,20 Microns Ltd.,933.6,35.7,125.2,4.51,845.1,121.5,45.1,1.78,NaN,NaN,423.6,70.6,NaN,NaN,NaN,NaN,2.5,61.9,6.3,NaN
3,2008-03-31,20 Microns Ltd.,1147.9,45.9,164.9,4.11,978.4,152.9,64.4,1.64,NaN,NaN,466.0,98.2,NaN,NaN,NaN,NaN,12.1,69.0,5.5,NaN
4,2009-03-31,20 Microns Ltd.,1454.8,13.4,144.7,4.38,1210.4,169.5,56.3,1.64,NaN,NaN,586.1,153.6,NaN,NaN,NaN,NaN,1.6,1.4,3.1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80089,2020-03-31,Zylog Systems Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80090,2021-03-31,Zylog Systems Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80091,2022-03-31,Zylog Systems Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80092,2023-03-31,Zylog Systems Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### 4 BVPS

In [11]:
bvps1 = pd.read_excel(rf"{supporting_folder_path}\Prowess IQ - BVPS.xlsx", sheet_name = 0, header = [0,1])
bvps2 = pd.read_excel(rf"{supporting_folder_path}\Prowess IQ - BVPS.xlsx", sheet_name = 1, header = [0,1])

In [12]:
bvps11 = bvps1.set_index([("Unnamed: 0_level_0","Company Name")])
bvps12 = bvps11.unstack().unstack(level = 1).reset_index()
bvps13 = bvps12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
bvps13["AsOnDate"] = pd.to_datetime(bvps13["AsOnDate"], format = "%b %Y" ) + MonthEnd(0)
bvps14 = bvps13.merge(compCodeNse, on = ["Company Name"], how = "left")

bvps21 = bvps2.set_index([("Unnamed: 0_level_0","Company Name")])
bvps22 = bvps21.unstack().unstack(level = 1).reset_index()
bvps23 = bvps22.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
bvps23["AsOnDate"] = pd.to_datetime(bvps23["AsOnDate"], format = "%b %Y" ) + MonthEnd(0)
bvps24 = bvps23.merge(compCodeNse, on = ["Company Name"], how = "left")

In [13]:
bvps = pd.concat([bvps14, bvps24], axis = 0).sort_values(by = ["Prowess company code", "AsOnDate"]).reset_index(drop= True).drop(["NSE symbol"], axis =1)
bvps = bvps[ bvps.columns[[1,0,2]]].dropna(subset = ["BV per Share "]).reset_index(drop = True).rename({"BV per Share ":"BVPS"}, axis = 1).drop_duplicates()
bvps

,Company Name,AsOnDate,BVPS
0,20 Microns Ltd.,2009-03-31,25.19
1,20 Microns Ltd.,2010-03-31,29.03
2,20 Microns Ltd.,2011-03-31,31.12
3,20 Microns Ltd.,2012-03-31,36.38
4,20 Microns Ltd.,2013-03-31,20.92
...,...,...,...
54469,Shree Karni Fabcom Ltd.,2024-03-31,0.00
54470,Pratham Epc Projects Ltd.,2024-03-31,0.00
54471,Signoria Creation Ltd.,2024-03-31,32.18
54472,Enfuse Solutions Ltd.,2024-03-31,35.07


#### 2 Deferred Tax Liability

In [14]:
defTaxLia1 = pd.read_excel(rf"{supporting_folder_path}\Prowess IQ - Deferred Tax Liability.xlsx", sheet_name = 0, header = [0,1])

In [15]:
defTaxLia11 = defTaxLia1.set_index([("Unnamed: 0_level_0","Company Name")])
defTaxLia12 = defTaxLia11.unstack().unstack(level = 1).reset_index()
defTaxLia13 = defTaxLia12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
defTaxLia13["AsOnDate"] = pd.to_datetime(defTaxLia13["AsOnDate"], format = "%b %Y" ) + MonthEnd(0)
defTaxLia14 = defTaxLia13.merge(compCodeNse, on = ["Company Name"], how = "left")

In [16]:
defTaxLia = defTaxLia14.sort_values(by = ["Prowess company code", "AsOnDate"]).reset_index(drop= True).drop([ "NSE symbol"], axis =1)
defTaxLia = defTaxLia[ defTaxLia.columns[[1,0,2]]].dropna(subset = ["Deferred tax liability"]).reset_index( drop = True).drop_duplicates()
defTaxLia

,Company Name,AsOnDate,Deferred tax liability
0,'K' Steamship Agencies Pvt. Ltd.,2011-03-31,7.3
1,'K' Steamship Agencies Pvt. Ltd.,2012-03-31,9.6
2,'K' Steamship Agencies Pvt. Ltd.,2013-03-31,0.6
3,'K' Steamship Agencies Pvt. Ltd.,2014-03-31,0.4
4,'K' Steamship Agencies Pvt. Ltd.,2015-03-31,0.3
...,...,...,...
204869,Economist Communications Ltd.,2023-03-31,0.4
204870,P V R Pictures Ltd.,2023-03-31,35.8
204871,Pankaj Piyush Trade & Invst. Ltd.,2023-03-31,0.2
204872,R D C Concrete (India) Pvt. Ltd.,2023-03-31,1.7


#### 3 Net Deferred Tax Liabilties

In [17]:
netDefTaxLia1 = pd.read_excel(rf"{supporting_folder_path}\Prowess IQ - Net Deferred Tax Liability.xlsx", sheet_name = 0, header = [0,1])

In [18]:
netDefTaxLia11 = netDefTaxLia1.set_index([("Unnamed: 0_level_0","Company Name")])
netDefTaxLia12 = netDefTaxLia11.unstack().unstack(level = 1).reset_index()
netDefTaxLia13 = netDefTaxLia12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
netDefTaxLia13["AsOnDate"] = pd.to_datetime(netDefTaxLia13["AsOnDate"], format = "%b %Y" ) + MonthEnd(0)
netDefTaxLia14 = netDefTaxLia13.merge(compCodeNse, on = ["Company Name"], how = "left")

In [19]:
netDefTaxLia = netDefTaxLia14.sort_values(by = ["Prowess company code", "AsOnDate"]).reset_index(drop= True).drop(["NSE symbol"], axis =1)
netDefTaxLia = netDefTaxLia[ netDefTaxLia.columns[[1,0,2]]].dropna(subset = ["Net deferred tax liabilities"]).reset_index( drop = True).drop_duplicates()
netDefTaxLia

,Company Name,AsOnDate,Net deferred tax liabilities
0,'K' Steamship Agencies Pvt. Ltd.,2011-03-31,6.1
1,'K' Steamship Agencies Pvt. Ltd.,2012-03-31,8.1
2,'K' Steamship Agencies Pvt. Ltd.,2013-03-31,-0.4
3,'K' Steamship Agencies Pvt. Ltd.,2014-03-31,-0.8
4,'K' Steamship Agencies Pvt. Ltd.,2015-03-31,-3.3
...,...,...,...
314437,Paras Healthcare Pvt. Ltd.,2023-03-31,-36.3
314438,R D C Concrete (India) Pvt. Ltd.,2023-03-31,-209.7
314439,S M C Comtrade Ltd.,2023-03-31,-6.1
314440,Shyama Infosys Ltd.,2023-03-31,-0.5


#### 5 Shares Outstanding

In [20]:
sharesOut1 = pd.read_excel(rf"{supporting_folder_path}\Prowess IQ - Shares Outstanding.xlsx", sheet_name = 0, header = [0,1])
sharesOut2 = pd.read_excel(rf"{supporting_folder_path}\Prowess IQ - Shares Outstanding.xlsx", sheet_name = 1, header = [0,1])

In [21]:
sharesOut11 = sharesOut1.set_index([("Unnamed: 0_level_0","Company Name")])
sharesOut12 = sharesOut11.unstack().unstack(level = 1).reset_index()
sharesOut13 = sharesOut12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
sharesOut13["AsOnDate"] = pd.to_datetime(sharesOut13["AsOnDate"], format = "%b %Y" ) + MonthEnd(0)
sharesOut14 = sharesOut13.merge(compCodeNse, on = ["Company Name"], how = "left")

sharesOut21 = sharesOut2.set_index([("Unnamed: 0_level_0","Company Name")])
sharesOut22 = sharesOut21.unstack().unstack(level = 1).reset_index()
sharesOut23 = sharesOut22.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
sharesOut23["AsOnDate"] = pd.to_datetime(sharesOut23["AsOnDate"], format = "%b %Y" ) + MonthEnd(0)
sharesOut24 = sharesOut23.merge(compCodeNse, on = ["Company Name"], how = "left")

In [22]:
sharesOut = pd.concat([sharesOut14, sharesOut24], axis = 0).sort_values(by = ["Prowess company code", "AsOnDate"]).reset_index(drop= True).drop(["NSE symbol"], axis =1)
sharesOut = sharesOut[ sharesOut.columns[[1,0,2]]].dropna(subset = ["Shares Outstanding "]).reset_index( drop = True).rename({"Shares Oustanding ":"Shares Oustanding"}, axis = 1).drop_duplicates()
sharesOut

,Company Name,AsOnDate,Shares Outstanding
0,20 Microns Ltd.,2009-03-31,14205248.0
1,20 Microns Ltd.,2010-03-31,14331028.0
2,20 Microns Ltd.,2011-03-31,14331028.0
3,20 Microns Ltd.,2012-03-31,14331028.0
4,20 Microns Ltd.,2013-03-31,31662056.0
...,...,...,...
54469,Shree Karni Fabcom Ltd.,2024-03-31,7072000.0
54470,Pratham Epc Projects Ltd.,2024-03-31,17760000.0
54471,Signoria Creation Ltd.,2024-03-31,4758000.0
54472,Enfuse Solutions Ltd.,2024-03-31,8847600.0


#### 6 Market Data

In [23]:
market1 = pd.read_excel(rf"{supporting_folder_path}\Prowess IQ - Market Data.xlsx", sheet_name = 0, header = [0,1])

In [24]:
market11 = market1.drop([("Unnamed: 1_level_0","NSE symbol")], axis = 1).set_index([("Unnamed: 0_level_0","Company Name")])
market12 = market11.unstack().unstack(level = 1).reset_index()
market13 = market12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
market13["AsOnDate"] = pd.to_datetime(market13["AsOnDate"], format = "%b %Y" ) + MonthEnd(0)

In [25]:
market = market13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Market Capitalisation", "Total Returns", "P/B ", "Market Capitalisation / Enterprise Value"], how = "all").reset_index(drop= True).drop_duplicates()
market

,AsOnDate,Company Name,Market Capitalisation,Total Returns,P/B,Market Capitalisation / Enterprise Value
0,2008-12-31,20 Microns Ltd.,220.98,2.62,0.42,0.33
1,2009-03-31,20 Microns Ltd.,214.50,0.67,0.60,0.28
2,2009-06-30,20 Microns Ltd.,328.85,-1.49,0.84,0.38
3,2009-09-30,20 Microns Ltd.,599.75,-1.99,1.47,0.52
4,2009-12-31,20 Microns Ltd.,659.94,-1.07,1.57,0.55
...,...,...,...,...,...,...
199881,2023-03-31,Zylog Systems Ltd.,20.65,-12.50,NaN,0.00
199882,2023-06-30,Zylog Systems Ltd.,20.65,-12.50,NaN,0.00
199883,2023-09-30,Zylog Systems Ltd.,20.65,-12.50,NaN,0.00
199884,2023-12-31,Zylog Systems Ltd.,20.65,-12.50,NaN,0.00


#### 7 Net Cash from investing activities

In [26]:
netCashInvest1 = pd.read_csv(rf"{supporting_folder_path}\Prowess IQ - Net Cash Investing activities.csv", header = [0,1])

In [27]:
netCashInvest11 = netCashInvest1.set_index([("Unnamed: 0_level_0","Company Name")])
netCashInvest12 = netCashInvest11.unstack().unstack(level = 1).reset_index()
netCashInvest13 = netCashInvest12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
netCashInvest13["AsOnDate"] = pd.to_datetime(netCashInvest13["AsOnDate"], format = "%b-%y" ) + MonthEnd(0)

In [28]:
netCashInvest = netCashInvest13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Net cash inflow or (outflow) from investing activities"], how = "all").reset_index(drop= True).drop_duplicates()
netCashInvest

,AsOnDate,Company Name,Net cash inflow or (outflow) from investing activities
0,2014-03-31,'K' Steamship Agencies Pvt. Ltd.,-96.3
1,2015-03-31,'K' Steamship Agencies Pvt. Ltd.,-181.9
2,2016-03-31,'K' Steamship Agencies Pvt. Ltd.,29.3
3,2017-03-31,'K' Steamship Agencies Pvt. Ltd.,-122.0
4,2018-03-31,'K' Steamship Agencies Pvt. Ltd.,-80.4
...,...,...,...
297232,2019-03-31,Zyxel Technology India Pvt. Ltd.,-0.2
297233,2020-03-31,Zyxel Technology India Pvt. Ltd.,-0.3
297234,2021-03-31,Zyxel Technology India Pvt. Ltd.,-0.2
297235,2022-03-31,Zyxel Technology India Pvt. Ltd.,-0.1


#### 8 Cash inflow from sale of fixed assets

In [29]:
inCashSaleFixed1 = pd.read_csv(rf"{supporting_folder_path}\Prowess IQ - Cash inflow from sale of fixed assets.csv", header = [0,1])

In [30]:
inCashSaleFixed11 = inCashSaleFixed1.set_index([("Unnamed: 0_level_0","Company Name")])
inCashSaleFixed12 = inCashSaleFixed11.unstack().unstack(level = 1).reset_index()
inCashSaleFixed13 = inCashSaleFixed12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
inCashSaleFixed13["AsOnDate"] = pd.to_datetime(inCashSaleFixed13["AsOnDate"], format = "%b-%y" ) + MonthEnd(0)

In [31]:
inCashSaleFixed = inCashSaleFixed13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Cash inflow due to sale of fixed assets"], how = "all").reset_index(drop= True).drop_duplicates()
inCashSaleFixed

,AsOnDate,Company Name,Cash inflow due to sale of fixed assets
0,2005-03-31,20 Microns Ltd.,0.3
1,2006-03-31,20 Microns Ltd.,0.4
2,2007-03-31,20 Microns Ltd.,0.9
3,2008-03-31,20 Microns Ltd.,2.8
4,2009-03-31,20 Microns Ltd.,4.2
...,...,...,...
91910,2019-03-31,Zydus Wellness Products Ltd.,995.3
91911,2020-03-31,Zydus Wellness Products Ltd.,0.5
91912,2021-03-31,Zydus Wellness Products Ltd.,20.1
91913,2022-03-31,Zydus Wellness Products Ltd.,6.2


#### 9 Cash outflow from Purchase of Fixed assets

In [32]:
outCashPurchaseFixed1 = pd.read_csv(rf"{supporting_folder_path}\Prowess IQ - Cash outflow from Purchase of Fixed assets.csv", header = [0,1])

In [33]:
outCashPurchaseFixed11 = outCashPurchaseFixed1.set_index([("Unnamed: 0_level_0","Company Name")])
outCashPurchaseFixed12 = outCashPurchaseFixed11.unstack().unstack(level = 1).reset_index()
outCashPurchaseFixed13 = outCashPurchaseFixed12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
outCashPurchaseFixed13["AsOnDate"] = pd.to_datetime(outCashPurchaseFixed13["AsOnDate"], format = "%b-%y" ) + MonthEnd(0)

In [34]:
outCashPurchaseFixed = outCashPurchaseFixed13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Cash (outflow) due to purchase of fixed assets"], how = "all").reset_index(drop= True).drop_duplicates()
outCashPurchaseFixed

,AsOnDate,Company Name,Cash (outflow) due to purchase of fixed assets
0,2014-03-31,'K' Steamship Agencies Pvt. Ltd.,-7.6
1,2015-03-31,'K' Steamship Agencies Pvt. Ltd.,-117.1
2,2016-03-31,'K' Steamship Agencies Pvt. Ltd.,-9.4
3,2017-03-31,'K' Steamship Agencies Pvt. Ltd.,-16.8
4,2018-03-31,'K' Steamship Agencies Pvt. Ltd.,-4.4
...,...,...,...
211570,2019-03-31,Zyxel Technology India Pvt. Ltd.,-0.2
211571,2020-03-31,Zyxel Technology India Pvt. Ltd.,-0.3
211572,2021-03-31,Zyxel Technology India Pvt. Ltd.,-0.3
211573,2022-03-31,Zyxel Technology India Pvt. Ltd.,-0.1


#### 10 Cash inflow from sale of intangible assets

In [35]:
inCashSaleIntangible1 = pd.read_csv(rf"{supporting_folder_path}\Prowess IQ - Cash inflow from sale of intangible assets.csv", header = [0,1])

In [36]:
inCashSaleIntangible11 = inCashSaleIntangible1.set_index([("Unnamed: 0_level_0","Company Name")])
inCashSaleIntangible12 = inCashSaleIntangible11.unstack().unstack(level = 1).reset_index()
inCashSaleIntangible13 = inCashSaleIntangible12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
inCashSaleIntangible13["AsOnDate"] = pd.to_datetime(inCashSaleIntangible13["AsOnDate"], format = "%b-%y" ) + MonthEnd(0)

In [37]:
inCashSaleIntangible = inCashSaleIntangible13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Cash inflow due to sale of Intangible assets"], how = "all").reset_index(drop= True).drop_duplicates()
inCashSaleIntangible

,AsOnDate,Company Name,Cash inflow due to sale of Intangible assets
0,2022-03-31,A B A Builders Ltd. [Merged],0.5
1,2019-03-31,A L M Infotech City Pvt. Ltd.,17.0
2,2016-03-31,Aarti International Ltd.,34.8
3,2021-03-31,Aarti International Ltd.,25.6
4,2023-03-31,Aarti International Ltd.,11.5
...,...,...,...
502,2016-03-31,Zydus Hospira Oncology Pvt. Ltd.,1018.7
503,2021-03-31,Zydus Hospira Oncology Pvt. Ltd.,0.4
504,2022-03-31,Zydus Hospira Oncology Pvt. Ltd.,0.5
505,2023-03-31,Zydus Hospira Oncology Pvt. Ltd.,0.3


#### 11 Cash outflow from Purchase of intangible assets

In [38]:
outCashPurchaseIntangible1 = pd.read_csv(rf"{supporting_folder_path}\Prowess IQ - Cash outflow from Purchase of intangible assets.csv", header = [0,1])

In [39]:
outCashPurchaseIntangible11 = outCashPurchaseIntangible1.set_index([("Unnamed: 0_level_0","Company Name")])
outCashPurchaseIntangible12 = outCashPurchaseIntangible11.unstack().unstack(level = 1).reset_index()
outCashPurchaseIntangible13 = outCashPurchaseIntangible12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
outCashPurchaseIntangible13["AsOnDate"] = pd.to_datetime(outCashPurchaseIntangible13["AsOnDate"], format = "%b-%y" ) + MonthEnd(0)

In [40]:
outCashPurchaseIntangible = outCashPurchaseIntangible13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Cash (outflow) due to purchase of Intangible assets"], how = "all").reset_index(drop= True).drop_duplicates()
outCashPurchaseIntangible

,AsOnDate,Company Name,Cash (outflow) due to purchase of Intangible assets
0,2024-03-31,360 One Prime Ltd.,-22.7
1,2017-03-31,3D Future Technologies Pvt. Ltd.,-1.7
2,2023-03-31,3D Future Technologies Pvt. Ltd.,-1.2
3,2023-03-31,7Seas Entertainment Ltd.,-5.5
4,2024-03-31,7Seas Entertainment Ltd.,-13.6
...,...,...,...
3969,2020-03-31,Zydus Hospira Oncology Pvt. Ltd.,-0.8
3970,2021-03-31,Zydus Hospira Oncology Pvt. Ltd.,-2.6
3971,2022-03-31,Zydus Hospira Oncology Pvt. Ltd.,-9.2
3972,2023-03-31,Zydus Hospira Oncology Pvt. Ltd.,-19.2


#### 12 Net change in cash and equivalents

In [41]:
netCashEquiv1 = pd.read_csv(rf"{supporting_folder_path}\CMIE - Net change in cash and equivalents.csv", header = [0,1])

In [42]:
netCashEquiv11 = netCashEquiv1.set_index([("Unnamed: 0_level_0","Company Name")])
netCashEquiv12 = netCashEquiv11.unstack().unstack(level = 1).reset_index()
netCashEquiv13 = netCashEquiv12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
netCashEquiv13["AsOnDate"] = pd.to_datetime(netCashEquiv13["AsOnDate"], format = "%b-%y" ) + MonthEnd(0)

In [43]:
netCashEquiv = netCashEquiv13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Net change in cash and cash equivalents"], how = "all").reset_index(drop= True).drop_duplicates()
netCashEquiv

,AsOnDate,Company Name,Net change in cash and cash equivalents
0,2022-03-31,1 Finance Pvt. Ltd.,3.00700
1,2023-03-31,1 Finance Pvt. Ltd.,164.62600
2,2024-03-31,1 Finance Pvt. Ltd.,-147.77300
3,2021-03-31,102 Mother Child Services (U P),-587.67300
4,2022-03-31,102 Mother Child Services (U P),75.21800
...,...,...,...
465199,2020-03-31,Zyxel Technology India Pvt. Ltd.,-1.34685
465200,2021-03-31,Zyxel Technology India Pvt. Ltd.,6.04053
465201,2022-03-31,Zyxel Technology India Pvt. Ltd.,-2.50261
465202,2023-03-31,Zyxel Technology India Pvt. Ltd.,39.68986


#### 13 Cash and cash equivalents as at the end of the year


In [44]:
cashEquivEndYear1 = pd.read_csv(rf"{supporting_folder_path}\CMIE - Cash and equivalents as at the end of the year.csv", header = [0,1])

In [45]:
cashEquivEndYear11 = cashEquivEndYear1.set_index([("Unnamed: 0_level_0","Company Name")])
cashEquivEndYear12 = cashEquivEndYear11.unstack().unstack(level = 1).reset_index()
cashEquivEndYear13 = cashEquivEndYear12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
cashEquivEndYear13["AsOnDate"] = pd.to_datetime(cashEquivEndYear13["AsOnDate"], format = "%b-%y" ) + MonthEnd(0)

In [46]:
cashEquivEndYear = cashEquivEndYear13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Cash and cash equivalents as at the end of the year"], how = "all").reset_index(drop= True).drop_duplicates()
cashEquivEndYear

,AsOnDate,Company Name,Cash and cash equivalents as at the end of the year
0,2022-03-31,1 Finance Pvt. Ltd.,3.00700
1,2023-03-31,1 Finance Pvt. Ltd.,167.63300
2,2024-03-31,1 Finance Pvt. Ltd.,19.86000
3,2014-03-31,10C India Internet Pvt. Ltd.,68.24976
4,2015-03-31,10C India Internet Pvt. Ltd.,170.68583
...,...,...,...
442006,2020-03-31,Zyxel Technology India Pvt. Ltd.,0.25544
442007,2021-03-31,Zyxel Technology India Pvt. Ltd.,6.29597
442008,2022-03-31,Zyxel Technology India Pvt. Ltd.,3.79335
442009,2023-03-31,Zyxel Technology India Pvt. Ltd.,43.48322


#### 14 Cash outflow from Purchase of intangible assets

In [47]:
cashEquivBegYear1 = pd.read_csv(rf"{supporting_folder_path}\CMIE - Cash at the beginning of the year.csv", header = [0,1])

In [48]:
cashEquivBegYear11 = cashEquivBegYear1.set_index([("Unnamed: 0_level_0","Company Name")])
cashEquivBegYear12 = cashEquivBegYear11.unstack().unstack(level = 1).reset_index()
cashEquivBegYear13 = cashEquivBegYear12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
cashEquivBegYear13["AsOnDate"] = pd.to_datetime(cashEquivBegYear13["AsOnDate"], format = "%b-%y" ) + MonthEnd(0)

In [49]:
cashEquivBegYear = cashEquivBegYear13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Cash and cash equivalents as at the beginning of the year"], how = "all").reset_index(drop= True).drop_duplicates()
cashEquivBegYear

,AsOnDate,Company Name,Cash and cash equivalents as at the beginning of the year
0,2022-03-31,1 Finance Pvt. Ltd.,0.00000
1,2023-03-31,1 Finance Pvt. Ltd.,3.00700
2,2024-03-31,1 Finance Pvt. Ltd.,167.63300
3,2021-03-31,102 Mother Child Services (U P),587.67300
4,2022-03-31,102 Mother Child Services (U P),-75.21800
...,...,...,...
467241,2020-03-31,Zyxel Technology India Pvt. Ltd.,1.60230
467242,2021-03-31,Zyxel Technology India Pvt. Ltd.,0.25544
467243,2022-03-31,Zyxel Technology India Pvt. Ltd.,6.29597
467244,2023-03-31,Zyxel Technology India Pvt. Ltd.,3.79335


## Intercorporate loans

#### 15 Current portion of long term inter-corporate loans

In [50]:
currLongInterCorpLoans1 = pd.read_csv(rf"{supporting_folder_path}\Intercorporate Loans\CMIE - Current portion of long term inter-corporate loans.csv", header = [0,1])

In [51]:
currLongInterCorpLoans11 = currLongInterCorpLoans1.set_index([("Unnamed: 0_level_0","Company Name")])
currLongInterCorpLoans12 = currLongInterCorpLoans11.unstack().unstack(level = 1).reset_index()
currLongInterCorpLoans13 = currLongInterCorpLoans12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
currLongInterCorpLoans13["AsOnDate"] = pd.to_datetime(currLongInterCorpLoans13["AsOnDate"], format = "%b-%y" ) + MonthEnd(0)

In [52]:
currLongInterCorpLoans = currLongInterCorpLoans13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Current portion of long term inter-corporate loans"], how = "all").reset_index(drop= True).drop_duplicates()
currLongInterCorpLoans

,AsOnDate,Company Name,Current portion of long term inter-corporate loans
0,2011-03-31,20 Microns Ltd.,3.0
1,2012-03-31,20 Microns Ltd.,3.6
2,2013-03-31,20 Microns Ltd.,0.7
3,2014-03-31,20 Microns Ltd.,3.1
4,2015-03-31,20 Microns Ltd.,6.7
...,...,...,...
1423,2012-03-31,Zydus Technologies Ltd. [Merged],66.7
1424,2014-03-31,Zydus Technologies Ltd. [Merged],0.0
1425,2015-03-31,Zydus Technologies Ltd. [Merged],114.7
1426,2018-03-31,Zydus Technologies Ltd. [Merged],1886.2


#### 16 Inter-corporate loans & borrowings (as per nbfc norms)

In [53]:
nbfcInterCorpLoans1 = pd.read_csv(rf"{supporting_folder_path}\Intercorporate Loans\CMIE - Inter-corporate loans & borrowings (as per nbfc norms).csv", header = [0,1])

In [54]:
nbfcInterCorpLoans11 = nbfcInterCorpLoans1.set_index([("Unnamed: 0_level_0","Company Name")])
nbfcInterCorpLoans12 = nbfcInterCorpLoans11.unstack().unstack(level = 1).reset_index()
nbfcInterCorpLoans13 = nbfcInterCorpLoans12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
nbfcInterCorpLoans13["AsOnDate"] = pd.to_datetime(nbfcInterCorpLoans13["AsOnDate"], format = "%b-%y" ) + MonthEnd(0)

In [55]:
nbfcInterCorpLoans = nbfcInterCorpLoans13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Inter-corporate loans & borrowings (as per nbfc norms)"], how = "all").reset_index(drop= True).drop_duplicates()
nbfcInterCorpLoans

,AsOnDate,Company Name,Inter-corporate loans & borrowings (as per nbfc norms)
0,2008-03-31,A C T Fininvest Ltd.,0.0
1,2009-03-31,A C T Fininvest Ltd.,1994.7
2,2016-03-31,A C T Fininvest Ltd.,558.5
3,2017-03-31,A C T Fininvest Ltd.,558.7
4,2018-03-31,A C T Fininvest Ltd.,2415.0
...,...,...,...
2275,2020-03-31,Zenith Credit Pvt. Ltd.,17.7
2276,2021-03-31,Zenith Credit Pvt. Ltd.,17.8
2277,2022-03-31,Zenith Credit Pvt. Ltd.,17.1
2278,2023-03-31,Zenith Credit Pvt. Ltd.,34.0


#### 17 Inter-corporate loans (Old Sch. VI)

In [56]:
oldSchInterCorpLoans1 = pd.read_csv(rf"{supporting_folder_path}\Intercorporate Loans\CMIE - Inter-corporate loans (Old Sch. VI).csv", header = [0,1])

In [57]:
oldSchInterCorpLoans11 = oldSchInterCorpLoans1.set_index([("Unnamed: 0_level_0","Company Name")])
oldSchInterCorpLoans12 = oldSchInterCorpLoans11.unstack().unstack(level = 1).reset_index()
oldSchInterCorpLoans13 = oldSchInterCorpLoans12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
oldSchInterCorpLoans13["AsOnDate"] = pd.to_datetime(oldSchInterCorpLoans13["AsOnDate"], format = "%b-%y" ) + MonthEnd(0)

In [58]:
oldSchInterCorpLoans = oldSchInterCorpLoans13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Inter-corporate loans (Old Sch. VI)"], how = "all").reset_index(drop= True).drop_duplicates()
oldSchInterCorpLoans

,AsOnDate,Company Name,Inter-corporate loans (Old Sch. VI)
0,2016-03-31,2 X 2 Logistics Pvt. Ltd.,186.5
1,2017-03-31,2 X 2 Logistics Pvt. Ltd.,211.4
2,2023-03-31,2 X 2 Logistics Pvt. Ltd.,80.0
3,2024-03-31,2 X 2 Logistics Pvt. Ltd.,80.0
4,2011-03-31,20 Microns Ltd.,7.3
...,...,...,...
107767,2020-03-31,Zyphar'S Pharmaceutics Pvt. Ltd.,33.6
107768,2021-03-31,Zyphar'S Pharmaceutics Pvt. Ltd.,6.3
107769,2022-03-31,Zyphar'S Pharmaceutics Pvt. Ltd.,30.2
107770,2023-03-31,Zyphar'S Pharmaceutics Pvt. Ltd.,23.2


#### 18 Long term inter-corporate loans

In [59]:
longInterCorpLoans1 = pd.read_csv(rf"{supporting_folder_path}\Intercorporate Loans\CMIE - Long term inter-corporate loans.csv", header = [0,1])

In [60]:
longInterCorpLoans11 = longInterCorpLoans1.set_index([("Unnamed: 0_level_0","Company Name")])
longInterCorpLoans12 = longInterCorpLoans11.unstack().unstack(level = 1).reset_index()
longInterCorpLoans13 = longInterCorpLoans12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
longInterCorpLoans13["AsOnDate"] = pd.to_datetime(longInterCorpLoans13["AsOnDate"], format = "%b-%y" ) + MonthEnd(0)

In [61]:
longInterCorpLoans = longInterCorpLoans13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Long term inter-corporate loans"], how = "all").reset_index(drop= True).drop_duplicates()
longInterCorpLoans

,AsOnDate,Company Name,Long term inter-corporate loans
0,2016-03-31,2 X 2 Logistics Pvt. Ltd.,186.5
1,2017-03-31,2 X 2 Logistics Pvt. Ltd.,211.4
2,2023-03-31,2 X 2 Logistics Pvt. Ltd.,80.0
3,2024-03-31,2 X 2 Logistics Pvt. Ltd.,80.0
4,2011-03-31,20 Microns Ltd.,7.3
...,...,...,...
43762,2020-03-31,Zyphar'S Pharmaceutics Pvt. Ltd.,33.6
43763,2021-03-31,Zyphar'S Pharmaceutics Pvt. Ltd.,6.3
43764,2022-03-31,Zyphar'S Pharmaceutics Pvt. Ltd.,30.2
43765,2023-03-31,Zyphar'S Pharmaceutics Pvt. Ltd.,23.2


#### 19 Secured inter-corporate loans (Old Sch. VI)

In [62]:
oldSchSecuredInterCorpLoans1 = pd.read_csv(rf"{supporting_folder_path}\Intercorporate Loans\CMIE - Secured inter-corporate loans (Old Sch. VI).csv", header = [0,1])

In [63]:
oldSchSecuredInterCorpLoans11 = oldSchSecuredInterCorpLoans1.set_index([("Unnamed: 0_level_0","Company Name")])
oldSchSecuredInterCorpLoans12 = oldSchSecuredInterCorpLoans11.unstack().unstack(level = 1).reset_index()
oldSchSecuredInterCorpLoans13 = oldSchSecuredInterCorpLoans12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
oldSchSecuredInterCorpLoans13["AsOnDate"] = pd.to_datetime(oldSchSecuredInterCorpLoans13["AsOnDate"], format = "%b-%y" ) + MonthEnd(0)

In [64]:
oldSchSecuredInterCorpLoans = oldSchSecuredInterCorpLoans13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Secured inter-corporate loans (Old Sch. VI)"], how = "all").reset_index(drop= True).drop_duplicates()
oldSchSecuredInterCorpLoans

,AsOnDate,Company Name,Secured inter-corporate loans (Old Sch. VI)
0,2022-03-31,20Cube Warehousing & Distribution Pvt. Ltd. [M...,29.2
1,2011-03-31,A 1 Century Trades Ltd.,21.5
2,2012-03-31,A 1 Century Trades Ltd.,13.0
3,2013-03-31,A 1 Century Trades Ltd.,13.0
4,2019-03-31,A 1 Century Trades Ltd.,61.9
...,...,...,...
4111,2020-03-31,Zuari International Ltd.,390.0
4112,2021-03-31,Zuari International Ltd.,1110.0
4113,2022-03-31,Zuari International Ltd.,150.0
4114,2023-03-31,Zuari International Ltd.,390.0


#### 20 Secured long term inter-corporate loans

In [65]:
securedLongInterCorpLoans1 = pd.read_csv(rf"{supporting_folder_path}\Intercorporate Loans\CMIE - Secured long term inter-corporate loans.csv", header = [0,1])

In [66]:
securedLongInterCorpLoans11 = securedLongInterCorpLoans1.set_index([("Unnamed: 0_level_0","Company Name")])
securedLongInterCorpLoans12 = securedLongInterCorpLoans11.unstack().unstack(level = 1).reset_index()
securedLongInterCorpLoans13 = securedLongInterCorpLoans12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
securedLongInterCorpLoans13["AsOnDate"] = pd.to_datetime(securedLongInterCorpLoans13["AsOnDate"], format = "%b-%y" ) + MonthEnd(0)

In [67]:
securedLongInterCorpLoans = securedLongInterCorpLoans13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Secured long term inter-corporate loans"], how = "all").reset_index(drop= True).drop_duplicates()
securedLongInterCorpLoans

,AsOnDate,Company Name,Secured long term inter-corporate loans
0,2022-03-31,20Cube Warehousing & Distribution Pvt. Ltd. [M...,23.0
1,2011-03-31,A 1 Century Trades Ltd.,21.5
2,2012-03-31,A 1 Century Trades Ltd.,13.0
3,2013-03-31,A 1 Century Trades Ltd.,13.0
4,2019-03-31,A 1 Century Trades Ltd.,61.9
...,...,...,...
5349,2024-03-31,York Exports Ltd.,12.2
5350,2020-03-31,Zar Jewels Pvt. Ltd.,35.9
5351,2021-03-31,Zar Jewels Pvt. Ltd.,37.5
5352,2022-03-31,Zar Jewels Pvt. Ltd.,34.7


#### 21 Secured short term inter-corporate loans

In [68]:
securedShortInterCorpLoans1 = pd.read_csv(rf"{supporting_folder_path}\Intercorporate Loans\CMIE - Secured short term inter-corporate loans.csv", header = [0,1])

In [69]:
securedShortInterCorpLoans11 = securedShortInterCorpLoans1.set_index([("Unnamed: 0_level_0","Company Name")])
securedShortInterCorpLoans12 = securedShortInterCorpLoans11.unstack().unstack(level = 1).reset_index()
securedShortInterCorpLoans13 = securedShortInterCorpLoans12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
securedShortInterCorpLoans13["AsOnDate"] = pd.to_datetime(securedShortInterCorpLoans13["AsOnDate"], format = "%b-%y" ) + MonthEnd(0)

In [70]:
securedShortInterCorpLoans = securedShortInterCorpLoans13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Secured short term inter-corporate loans"], how = "all").reset_index(drop= True).drop_duplicates()
securedShortInterCorpLoans

,AsOnDate,Company Name,Secured short term inter-corporate loans
0,2022-03-31,20Cube Warehousing & Distribution Pvt. Ltd. [M...,6.2
1,2020-03-31,A A Infraproperties Pvt. Ltd. [Merged],4927.2
2,2021-03-31,A A Infraproperties Pvt. Ltd. [Merged],6461.3
3,2022-03-31,A A Infraproperties Pvt. Ltd. [Merged],7124.4
4,2023-03-31,A A Infraproperties Pvt. Ltd. [Merged],7169.2
...,...,...,...
1059,2020-03-31,Zuari International Ltd.,390.0
1060,2021-03-31,Zuari International Ltd.,1110.0
1061,2022-03-31,Zuari International Ltd.,150.0
1062,2023-03-31,Zuari International Ltd.,390.0


#### 22 Short term inter-corporate loans

In [71]:
shortInterCorpLoans1 = pd.read_csv(rf"{supporting_folder_path}\Intercorporate Loans\CMIE - Short term inter-corporate loans.csv", header = [0,1])

In [72]:
shortInterCorpLoans11 = shortInterCorpLoans1.set_index([("Unnamed: 0_level_0","Company Name")])
shortInterCorpLoans12 = shortInterCorpLoans11.unstack().unstack(level = 1).reset_index()
shortInterCorpLoans13 = shortInterCorpLoans12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
shortInterCorpLoans13["AsOnDate"] = pd.to_datetime(shortInterCorpLoans13["AsOnDate"], format = "%b-%y" ) + MonthEnd(0)

In [73]:
shortInterCorpLoans = shortInterCorpLoans13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Short term inter-corporate loans"], how = "all").reset_index(drop= True).drop_duplicates()
shortInterCorpLoans

,AsOnDate,Company Name,Short term inter-corporate loans
0,2012-03-31,20 Microns Ltd.,15.3
1,2013-03-31,20 Microns Ltd.,6.8
2,2015-03-31,20 Microns Ltd.,18.4
3,2017-03-31,20 Microns Ltd.,3.6
4,2024-03-31,20 Microns Ltd.,0.3
...,...,...,...
35149,2012-03-31,Zylog Systems Ltd.,195.2
35150,2013-03-31,Zylog Systems Ltd.,957.2
35151,2014-03-31,Zylog Systems Ltd.,964.7
35152,2015-03-31,Zylog Systems Ltd.,985.1


#### 23 Unsecured inter-corporate loans (Old Sch. VI)

In [74]:
oldUnsecuredInterCorpLoans1 = pd.read_csv(rf"{supporting_folder_path}\Intercorporate Loans\CMIE - Unsecured inter-corporate loans (Old Sch. VI).csv", header = [0,1])

In [75]:
oldUnsecuredInterCorpLoans11 = oldUnsecuredInterCorpLoans1.set_index([("Unnamed: 0_level_0","Company Name")])
oldUnsecuredInterCorpLoans12 = oldUnsecuredInterCorpLoans11.unstack().unstack(level = 1).reset_index()
oldUnsecuredInterCorpLoans13 = oldUnsecuredInterCorpLoans12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
oldUnsecuredInterCorpLoans13["AsOnDate"] = pd.to_datetime(oldUnsecuredInterCorpLoans13["AsOnDate"], format = "%b-%y" ) + MonthEnd(0)

In [76]:
oldUnsecuredInterCorpLoans = oldUnsecuredInterCorpLoans13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Unsecured inter-corporate loans (Old Sch. VI)"], how = "all").reset_index(drop= True).drop_duplicates()
oldUnsecuredInterCorpLoans

,AsOnDate,Company Name,Unsecured inter-corporate loans (Old Sch. VI)
0,2023-03-31,2 X 2 Logistics Pvt. Ltd.,80.0
1,2024-03-31,2 X 2 Logistics Pvt. Ltd.,80.0
2,2011-03-31,20 Microns Ltd.,7.3
3,2012-03-31,20 Microns Ltd.,22.7
4,2013-03-31,20 Microns Ltd.,37.0
...,...,...,...
101246,2020-03-31,Zyphar'S Pharmaceutics Pvt. Ltd.,33.6
101247,2021-03-31,Zyphar'S Pharmaceutics Pvt. Ltd.,6.3
101248,2022-03-31,Zyphar'S Pharmaceutics Pvt. Ltd.,30.2
101249,2023-03-31,Zyphar'S Pharmaceutics Pvt. Ltd.,23.2


#### 24 Unsecured long term inter-corporate loans

In [77]:
unsecuredLongInterCorpLoans1 = pd.read_csv(rf"{supporting_folder_path}\Intercorporate Loans\CMIE - Unsecured long term inter-corporate loans.csv", header = [0,1])

In [78]:
unsecuredLongInterCorpLoans11 = unsecuredLongInterCorpLoans1.set_index([("Unnamed: 0_level_0","Company Name")])
unsecuredLongInterCorpLoans12 = unsecuredLongInterCorpLoans11.unstack().unstack(level = 1).reset_index()
unsecuredLongInterCorpLoans13 = unsecuredLongInterCorpLoans12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
unsecuredLongInterCorpLoans13["AsOnDate"] = pd.to_datetime(unsecuredLongInterCorpLoans13["AsOnDate"], format = "%b-%y" ) + MonthEnd(0)

In [79]:
unsecuredLongInterCorpLoans = unsecuredLongInterCorpLoans13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Unsecured long term inter-corporate loans"], how = "all").reset_index(drop= True).drop_duplicates()
unsecuredLongInterCorpLoans

,AsOnDate,Company Name,Unsecured long term inter-corporate loans
0,2023-03-31,2 X 2 Logistics Pvt. Ltd.,80.0
1,2024-03-31,2 X 2 Logistics Pvt. Ltd.,80.0
2,2011-03-31,20 Microns Ltd.,7.3
3,2012-03-31,20 Microns Ltd.,7.4
4,2013-03-31,20 Microns Ltd.,30.2
...,...,...,...
41729,2020-03-31,Zyphar'S Pharmaceutics Pvt. Ltd.,33.6
41730,2021-03-31,Zyphar'S Pharmaceutics Pvt. Ltd.,6.3
41731,2022-03-31,Zyphar'S Pharmaceutics Pvt. Ltd.,30.2
41732,2023-03-31,Zyphar'S Pharmaceutics Pvt. Ltd.,23.2


#### 25 Unsecured short term inter-corporate loans

In [80]:
unsecuredShortInterCorpLoans1 = pd.read_csv(rf"{supporting_folder_path}\Intercorporate Loans\CMIE - Unsecured short term inter-corporate loans.csv", header = [0,1])

In [81]:
unsecuredShortInterCorpLoans11 = unsecuredShortInterCorpLoans1.set_index([("Unnamed: 0_level_0","Company Name")])
unsecuredShortInterCorpLoans12 = unsecuredShortInterCorpLoans11.unstack().unstack(level = 1).reset_index()
unsecuredShortInterCorpLoans13 = unsecuredShortInterCorpLoans12.rename({"level_0":"AsOnDate", ("Unnamed: 0_level_0","Company Name"):"Company Name"}, axis = 1).sort_values(by = ["Company Name", "AsOnDate"]).reset_index(drop = True)
unsecuredShortInterCorpLoans13["AsOnDate"] = pd.to_datetime(unsecuredShortInterCorpLoans13["AsOnDate"], format = "%b-%y" ) + MonthEnd(0)

In [82]:
unsecuredShortInterCorpLoans = unsecuredShortInterCorpLoans13.sort_values(by = ["Company Name", "AsOnDate"]).dropna(subset = ["Unsecured short term inter-corporate loans"], how = "all").reset_index(drop= True).drop_duplicates()
unsecuredShortInterCorpLoans

,AsOnDate,Company Name,Unsecured short term inter-corporate loans
0,2011-03-31,20 Microns Ltd.,0.0
1,2012-03-31,20 Microns Ltd.,15.3
2,2013-03-31,20 Microns Ltd.,6.8
3,2014-03-31,20 Microns Ltd.,0.0
4,2015-03-31,20 Microns Ltd.,18.4
...,...,...,...
34462,2012-03-31,Zylog Systems Ltd.,195.2
34463,2013-03-31,Zylog Systems Ltd.,957.2
34464,2014-03-31,Zylog Systems Ltd.,964.7
34465,2015-03-31,Zylog Systems Ltd.,985.1


#### 26 Entity ID and stuff

In [83]:
# iden = fin_identifiers[fin_identifiers.columns[[0,1,2,3,4,5,6,7,8,9,10,11,12,13]]].droplevel( 0, axis =1)

In [84]:
# iden

In [85]:
iden = pd.read_csv(rf"{supporting_folder_path}\Prowess IQ - Indicators 2.txt", sep="|", dtype = {"NIC code":str})

In [86]:
iden

,Company Name,Prowess company code,Entity type,Entity type code,Incorporation year,Industry group,Industry group code,NIC name,NIC code,Ownership group code,Ownership group,Age group,NSE symbol,Head office address
0,'K' Steamship Agencies Pvt. Ltd.,3,Private Ltd.,10203010000,1971.0,Diversified,1.100000e+14,Diversified,34,20102000000,Private (Indian),Between 1951 and 1971,NaN,NaN
1,'X'Clusive Business Centre Pvt. Ltd.,307865,Private Ltd.,10203010000,1996.0,Diversified financial services,1.022000e+14,"Other financial service activities, except ins...",649,20102000000,Private (Indian),Between 1991 and 2013,NaN,NaN
2,1 To 1 Help.Net Pvt. Ltd.,591675,Private Ltd.,10203010000,2001.0,Business services & consultancy,1.010415e+14,Management consultancy activities,70200,20102000000,Private (Indian),Between 1991 and 2013,NaN,NaN
3,10C India Internet Pvt. Ltd.,556976,Private Ltd.,10203010000,2012.0,Telecommunication services,1.010406e+14,Other telecommunications activities,61900,20102000000,Private (Indian),Between 1991 and 2013,NaN,NaN
4,10I Commerce Services Pvt. Ltd.,560502,Private Ltd.,10203010000,2015.0,Wholesale trading,1.010404e+14,"Wholesale of radio, television and other consu...",46522,20102000000,Private (Indian),After 2014,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55316,Zylog Systems Ltd.,275793,Public Ltd.,10203020000,1995.0,Computer software,1.010408e+14,Providing software support and maintenance to ...,62013,20102000000,Private (Indian),Between 1991 and 2013,ZYLOG,NaN
55317,Zyma Laboratories Ltd. [Merged],275794,Public Ltd.,10203020000,1970.0,Other fund based financial services,1.020400e+14,"Trusts, funds and other financial vehicles",64300,20102000000,Private (Indian),Between 1951 and 1971,NaN,NaN
55318,Zyphar'S Pharmaceutics Pvt. Ltd.,565342,Private Ltd.,10203010000,2006.0,Wholesale trading,1.010404e+14,Wholesale of pharmaceutical and medical goods,46497,20102000000,Private (Indian),Between 1991 and 2013,NaN,NaN
55319,Zytel Agencies Ltd.,275795,Public Ltd.,10203020000,1984.0,Other fund based financial services,1.020400e+14,"Trusts, funds and other financial vehicles",64300,20102000000,Private (Indian),Between 1972 and 1985,NaN,NaN


### Merging all the raw Data together before merging with main

In [87]:
# 1 fin
# 2 defTaxLia
# 3 netDefTaxLia

# 4 bvps
# 5 sharesOut

# 6 market

# 7 netCashInvest
# 8 inCashSaleFixed
# 9 outCashPurchaseFixed
# 10 inCashSaleIntangible
# 11 outCashPurchaseIntangible

# 12 netCashEquiv
# 13 cashEquivEndYear
# 14 cashEquivBegYear

# 15 currLongInterCorpLoans
# 16 nbfcInterCorpLoans
# 17 oldSchInterCorpLoans
# 18 longInterCorpLoans
# 19 oldSchSecuredInterCorpLoans
# 20 securedLongInterCorpLoans
# 21 securedShortInterCorpLoans
# 22 shortInterCorpLoans
# 23 oldUnsecuredInterCorpLoans
# 24 unsecuredLongInterCorpLoans
# 25 unsecuredShortInterCorpLoans

# 26 iden
# 27 own

In [88]:
# raw1 = compCodeNse.merge(fin, on = ["Company Name"], how = "outer")
raw2 = fin.merge(defTaxLia, on = ["Company Name", "AsOnDate"], how = "outer")
raw3 = raw2.merge(netDefTaxLia, on = ["Company Name", "AsOnDate"], how = "outer")

In [89]:
raw4 = raw3.merge(bvps, on = ["Company Name", "AsOnDate"], how = "outer")
raw5 = raw4.merge(sharesOut, on = ["Company Name", "AsOnDate"], how = "outer")

raw6 = raw5.merge(market, on = ["Company Name", "AsOnDate"], how = "outer")

raw7 = raw6.merge(netCashInvest, on = ["Company Name", "AsOnDate"], how = "outer")

raw8 = raw7.merge(inCashSaleFixed, on = ["Company Name", "AsOnDate"], how = "outer")
raw9 = raw8.merge(outCashPurchaseFixed, on = ["Company Name", "AsOnDate"], how = "outer")

raw10 = raw9.merge(inCashSaleIntangible, on = ["Company Name", "AsOnDate"], how = "outer")
raw11 = raw10.merge(outCashPurchaseIntangible, on = ["Company Name", "AsOnDate"], how = "outer")

raw12 = raw11.merge(netCashEquiv, on = ["Company Name", "AsOnDate"], how = "outer")
raw13 = raw12.merge(cashEquivEndYear, on = ["Company Name", "AsOnDate"], how = "outer")
raw14 = raw13.merge(cashEquivBegYear, on = ["Company Name", "AsOnDate"], how = "outer")

raw15 = raw14.merge(currLongInterCorpLoans, on = ["Company Name", "AsOnDate"], how = "outer")
raw16 = raw15.merge(nbfcInterCorpLoans, on = ["Company Name", "AsOnDate"], how = "outer")
raw17 = raw16.merge(oldSchInterCorpLoans, on = ["Company Name", "AsOnDate"], how = "outer")
raw18 = raw17.merge(longInterCorpLoans, on = ["Company Name", "AsOnDate"], how = "outer")
raw19 = raw18.merge(oldSchSecuredInterCorpLoans, on = ["Company Name", "AsOnDate"], how = "outer")
raw20 = raw19.merge(securedLongInterCorpLoans, on = ["Company Name", "AsOnDate"], how = "outer")
raw21 = raw20.merge(securedShortInterCorpLoans, on = ["Company Name", "AsOnDate"], how = "outer")
raw22 = raw21.merge(shortInterCorpLoans, on = ["Company Name", "AsOnDate"], how = "outer")
raw23 = raw22.merge(oldUnsecuredInterCorpLoans, on = ["Company Name", "AsOnDate"], how = "outer")
raw24 = raw23.merge(unsecuredLongInterCorpLoans, on = ["Company Name", "AsOnDate"], how = "outer")
raw25 = raw24.merge(unsecuredShortInterCorpLoans, on = ["Company Name", "AsOnDate"], how = "outer")

raw25 = raw24.merge(iden, on = ["Company Name"], how = "outer")
raw26 = raw25.merge(own, on = ["Company Name", "AsOnDate"], how = "outer")

In [90]:
raw26

,AsOnDate,Company Name,Sales,Profit after tax,PBDITA,Return on total assets,Total assets,Operating profit of non-financial companies,Operating profit of financial companies,Debt to equity ratio (times),Long term borrowings incl current portion,Long term borrowings excl current portion,Debt,Trade payables (Old Sch. VI),Subscribed preference capital,Paid up preference capital (net of forfeited preference capital) (Old Sch. VI),Paid up preference capital (net of forfeited preference capital),Paid up preference shares,Total other receivables,Net cash flow from operating activities,Research & development expenses,Inter-corporate loans (Old Sch. VI)_x,Deferred tax liability,Net deferred tax liabilities,BVPS,Shares Outstanding,Market Capitalisation,Total Returns,P/B,Market Capitalisation / Enterprise Value,Net cash inflow or (outflow) from investing activities,Cash inflow due to sale of fixed assets,Cash (outflow) due to purchase of fixed assets,Cash inflow due to sale of Intangible assets,Cash (outflow) due to purchase of Intangible assets,Net change in cash and cash equivalents,Cash and cash equivalents as at the end of the year,Cash and cash equivalents as at the beginning of the year,Current portion of long term inter-corporate loans,Inter-corporate loans & borrowings (as per nbfc norms),Inter-corporate loans (Old Sch. VI)_y,Long term inter-corporate loans,Secured inter-corporate loans (Old Sch. VI),Secured long term inter-corporate loans,Secured short term inter-corporate loans,Short term inter-corporate loans,Unsecured inter-corporate loans (Old Sch. VI),Unsecured long term inter-corporate loans,Prowess company code,Entity type,Entity type code,Incorporation year,Industry group,Industry group code,NIC name,NIC code,Ownership group code,Ownership group,Age group,NSE symbol,Head office address,Total Shares (In %) - Shares held,Promoters (In %) - Shares held,Indian Promoters (In %) - Shares held,Foreign Promoters (In %) - Shares held,Non-promoters (In %) - Shares held,Non-promoter Institutions (In %) - Shares held,Non-promoter Non-institutions (In %) - Shares held
0,2011-03-31,'K' Steamship Agencies Pvt. Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.3,6.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,Private Ltd.,1.020301e+10,1971.0,Diversified,1.100000e+14,Diversified,34,2.010200e+10,Private (Indian),Between 1951 and 1971,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012-03-31,'K' Steamship Agencies Pvt. Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.6,8.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,Private Ltd.,1.020301e+10,1971.0,Diversified,1.100000e+14,Diversified,34,2.010200e+10,Private (Indian),Between 1951 and 1971,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2013-03-31,'K' Steamship Agencies Pvt. Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.6,-0.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,Private Ltd.,1.020301e+10,1971.0,Diversified,1.100000e+14,Diversified,34,2.010200e+10,Private (Indian),Between 1951 and 1971,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2014-03-31,'K' Steamship Agencies Pvt. Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.4,-0.8,NaN,NaN,NaN,NaN,NaN,NaN,-96.3,NaN,-7.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,Private Ltd.,1.020301e+10,1971.0,Diversified,1.100000e+14,Diversified,34,2.010200e+10,Private (Indian),Between 1951 and 1971,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-03-31,'K' Steamship Agencies Pvt. Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.3,-3.3,NaN,NaN,NaN,NaN,NaN,NaN,-181.9,NaN,-117.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,Private Ltd.,1.020301e+10,1971.0,Diversified,1.100000e+14,Diversified,34,2.01

In [91]:
finance_data = raw26.copy()
finance_data.to_pickle(rf"{output_folder_path}\Finance_data_merged_06_24.pkl")

In [92]:
main = pd.read_pickle(rf"{import_folder_path}\Main_Firm_No Loc_No Fin.pkl")
main

,Symbol,AsOnDate,TotDirCount,AsOnYear,LnBoardSize,FemaleDirCount,PercentWomenDir,Women_dummy,AvgAge,CountRookie,Rookie_dummy,CountNonRookie,CountIndep,PercentIndep,CountNonIndep,CountRookieIndep,CountRookieNonIndep,CountNonRookieIndep,CountNonRookieNonIndep,CountPromoterDir,CountBusyDir,PercentBusyDir,Busy_dummy,HasCeoMD,HasChairman,HasPromoterOnBoard,HasDualityChairmanMD,IsFamilyManaged,HasFamilyChairman,HasFamilyChairmanAndCEO,CountFirstTermDir,CountFirstTermIndepDir,PercentFirstTermIndepDir,CountTurnOverIndepDir,HasIndepTurnOver,AvgTenureInYearsinCompTotal,AvgTenureInYearsinCompIndep,PercentRookieIndep,PercentNonRookieIndep,PercentRookieNonIndep,PercentNonRookieNonIndep,PercentRookie,PercentNonRookie,IsRookieBoard,JustOneRookie,TwoOrMoreRookies,CountMBA,PercentMBA,CountIndepMBA,PercentIndepMBA,CountRookieIndepMBA,PercentRookieIndepMBA,CountPhD,PercentPhD,CountIndepPhD,PercentIndepPhD,CountRookieIndepPhD,PercentRookieIndepPhD,CountFinanceXP,PercentFinanceXP,CountIndepFinanceXP,PercentIndepFinanceXP,CountRookieIndepFinanceXP,PercentRookieIndepFinanceXP,CountTechXP,PercentTechXP,CountIndepTechXP,PercentIndepTechXP,CountRookieIndepTechXP,PercentRookieIndepTechXP,CountFinanceSkill,PercentFinanceSkill,CountIndepFinanceSkill,PercentIndepFinanceSkill,CountRookieIndepFinanceSkill,PercentRookieIndepFinanceSkill,CountTechSkill,PercentTechSkill,CountIndepTechSkill,PercentIndepTechSkill,CountRookieIndepTechSkill,PercentRookieIndepTechSkill,CountRelatedIndustryXP,PercentRelatedIndustryXP,CountIndepRelatedIndustryXP,PercentIndepRelatedIndustryXP,CountRookieIndepRelatedIndustryXP,PercentRookieIndepRelatedIndustryXP,CountExecXP,PercentExecXP,CountIndepExecXP,PercentIndepExecXP,CountRookieIndepExecXP,PercentRookieIndepExecXP,CountPublicExecXP,PercentPublicExecXP,CountIndepPublicExecXP,PercentIndepPublicExecXP,CountRookieIndepPublicExecXP,PercentRookieIndepPublicExecXP,CountPrivateExecXP,PercentPrivateExecXP,CountIndepPrivateExecXP,PercentIndepPrivateExecXP,CountRookieIndepPrivateExecXP,PercentRookieIndepPrivateExecXP,CountCeoMDChairXP,PercentCeoMDChairXP,CountIndepCeoMDChairXP,PercentIndepCeoMDChairXP,CountRookieIndepCeoMDChairXP,PercentRookieIndepCeoMDChairXP,CountRookieAppoint,CountRookieIndepAppoint,CountNonRookieAppoint,CountNonRookieIndepAppoint,RookieAppointDummy,RookieIndepAppointDummy,NonRookieAppointDummy,NonRookieIndepAppointDummy,TotalDirCess,TotalRookieIndepDirCess,TotalDirAppoint,TotalRookieIndepDirAppoint,UniqueSkills,CountUniqueSkills,CountRetiringTermLimit,PercentRetiringTermLimit,TermLimitRetireesPcodeList,CountDirTurnOver_exits_retire,PercentDirTurnOver_exits_retire,CountDirTurnOver_real_exits,PercentDirTurnOver_real_exits,CountIndepDirTurnOver_exits_retire,PercentIndepDirTurnOver_exits_retire,CountIndepDirTurnOver_real_exits,PercentIndepDirTurnOver_real_exits,ZeroYearPCodeList,FirstYearPCodeList,TwoYearPCodeList,ThreeYearPCodeList,PCodeList,ZeroYearIndepPCodeList,FirstYearIndepPCodeList,TwoYearIndepPCodeList,ThreeYearIndepPCodeList,IndepPCodeList,BoardAbsenceMean,Academic_mean,Company Business_mean,combined_sustainability_mean,combined_entrepreneurial_mean,combined_compensation_mean,combined_conglomerate_experience_mean,combined_hr_mean,combined_technology_mean,combined_finance_accounting_mean,combined_governance_mean,combined_government_policy_mean,combined_international_mean,combined_leadership_mean,combined_legal_mean,combined_marketing_mean,combined_risk_management_mean,combined_scientific_mean,combined_strategic_planning_mean,combined_manufacturing_supply_chain_mean,combined_leadership_outside_board_mean,skill_committee_sustainability_mean,skill_committee_entrepreneurial_mean,skill_committee_compensation_mean,skill_committee_conglomerate_experience_mean,skill_committee_hr_mean,skill_committee_technology_mean,skill_committee_finance_accounting_mean,skill_committee_governance_mean,skill_committee_government_policy_mean,skill_committee_international_mean,skill_committee_leadership_mean,skill_committee_legal_mean,skill_co

In [93]:
merged = main.merge(finance_data, left_on = ["Symbol", "AsOnDate"], right_on = ["NSE symbol", "AsOnDate"], how = "left")

In [94]:
FirmID = merged.Symbol.drop_duplicates().reset_index(drop = True).reset_index().rename({"index":"FirmID"}, axis = 1)
FirmID

,FirmID,Symbol
0,0,20MICRONS
1,1,21STCENMGM
2,2,360ONE
3,3,3IINFOLTD
4,4,3MINDIA
...,...,...
2775,2775,ZUARI
2776,2776,ZUARIIND
2777,2777,ZYDUSLIFE
2778,2778,ZYDUSWELL


In [95]:
merged2 = FirmID.merge(merged, on = "Symbol", how = "inner")
merged2

,FirmID,Symbol,AsOnDate,TotDirCount,AsOnYear,LnBoardSize,FemaleDirCount,PercentWomenDir,Women_dummy,AvgAge,CountRookie,Rookie_dummy,CountNonRookie,CountIndep,PercentIndep,CountNonIndep,CountRookieIndep,CountRookieNonIndep,CountNonRookieIndep,CountNonRookieNonIndep,CountPromoterDir,CountBusyDir,PercentBusyDir,Busy_dummy,HasCeoMD,HasChairman,HasPromoterOnBoard,HasDualityChairmanMD,IsFamilyManaged,HasFamilyChairman,HasFamilyChairmanAndCEO,CountFirstTermDir,CountFirstTermIndepDir,PercentFirstTermIndepDir,CountTurnOverIndepDir,HasIndepTurnOver,AvgTenureInYearsinCompTotal,AvgTenureInYearsinCompIndep,PercentRookieIndep,PercentNonRookieIndep,PercentRookieNonIndep,PercentNonRookieNonIndep,PercentRookie,PercentNonRookie,IsRookieBoard,JustOneRookie,TwoOrMoreRookies,CountMBA,PercentMBA,CountIndepMBA,PercentIndepMBA,CountRookieIndepMBA,PercentRookieIndepMBA,CountPhD,PercentPhD,CountIndepPhD,PercentIndepPhD,CountRookieIndepPhD,PercentRookieIndepPhD,CountFinanceXP,PercentFinanceXP,CountIndepFinanceXP,PercentIndepFinanceXP,CountRookieIndepFinanceXP,PercentRookieIndepFinanceXP,CountTechXP,PercentTechXP,CountIndepTechXP,PercentIndepTechXP,CountRookieIndepTechXP,PercentRookieIndepTechXP,CountFinanceSkill,PercentFinanceSkill,CountIndepFinanceSkill,PercentIndepFinanceSkill,CountRookieIndepFinanceSkill,PercentRookieIndepFinanceSkill,CountTechSkill,PercentTechSkill,CountIndepTechSkill,PercentIndepTechSkill,CountRookieIndepTechSkill,PercentRookieIndepTechSkill,CountRelatedIndustryXP,PercentRelatedIndustryXP,CountIndepRelatedIndustryXP,PercentIndepRelatedIndustryXP,CountRookieIndepRelatedIndustryXP,PercentRookieIndepRelatedIndustryXP,CountExecXP,PercentExecXP,CountIndepExecXP,PercentIndepExecXP,CountRookieIndepExecXP,PercentRookieIndepExecXP,CountPublicExecXP,PercentPublicExecXP,CountIndepPublicExecXP,PercentIndepPublicExecXP,CountRookieIndepPublicExecXP,PercentRookieIndepPublicExecXP,CountPrivateExecXP,PercentPrivateExecXP,CountIndepPrivateExecXP,PercentIndepPrivateExecXP,CountRookieIndepPrivateExecXP,PercentRookieIndepPrivateExecXP,CountCeoMDChairXP,PercentCeoMDChairXP,CountIndepCeoMDChairXP,PercentIndepCeoMDChairXP,CountRookieIndepCeoMDChairXP,PercentRookieIndepCeoMDChairXP,CountRookieAppoint,CountRookieIndepAppoint,CountNonRookieAppoint,CountNonRookieIndepAppoint,RookieAppointDummy,RookieIndepAppointDummy,NonRookieAppointDummy,NonRookieIndepAppointDummy,TotalDirCess,TotalRookieIndepDirCess,TotalDirAppoint,TotalRookieIndepDirAppoint,UniqueSkills,CountUniqueSkills,CountRetiringTermLimit,PercentRetiringTermLimit,TermLimitRetireesPcodeList,CountDirTurnOver_exits_retire,PercentDirTurnOver_exits_retire,CountDirTurnOver_real_exits,PercentDirTurnOver_real_exits,CountIndepDirTurnOver_exits_retire,PercentIndepDirTurnOver_exits_retire,CountIndepDirTurnOver_real_exits,PercentIndepDirTurnOver_real_exits,ZeroYearPCodeList,FirstYearPCodeList,TwoYearPCodeList,ThreeYearPCodeList,PCodeList,ZeroYearIndepPCodeList,FirstYearIndepPCodeList,TwoYearIndepPCodeList,ThreeYearIndepPCodeList,IndepPCodeList,BoardAbsenceMean,Academic_mean,Company Business_mean,combined_sustainability_mean,combined_entrepreneurial_mean,combined_compensation_mean,combined_conglomerate_experience_mean,combined_hr_mean,combined_technology_mean,combined_finance_accounting_mean,combined_governance_mean,combined_government_policy_mean,combined_international_mean,combined_leadership_mean,combined_legal_mean,combined_marketing_mean,combined_risk_management_mean,combined_scientific_mean,combined_strategic_planning_mean,combined_manufacturing_supply_chain_mean,combined_leadership_outside_board_mean,skill_committee_sustainability_mean,skill_committee_entrepreneurial_mean,skill_committee_compensation_mean,skill_committee_conglomerate_experience_mean,skill_committee_hr_mean,skill_committee_technology_mean,skill_committee_finance_accounting_mean,skill_committee_governance_mean,skill_committee_government_policy_mean,skill_committee_international_mean,skill_committee_leadership_mean,skill_committee_legal_mean,s

In [96]:
merged2["BVCommonStock"] = merged2["BVPS"] * merged2["Shares Outstanding "]
merged2

,FirmID,Symbol,AsOnDate,TotDirCount,AsOnYear,LnBoardSize,FemaleDirCount,PercentWomenDir,Women_dummy,AvgAge,CountRookie,Rookie_dummy,CountNonRookie,CountIndep,PercentIndep,CountNonIndep,CountRookieIndep,CountRookieNonIndep,CountNonRookieIndep,CountNonRookieNonIndep,CountPromoterDir,CountBusyDir,PercentBusyDir,Busy_dummy,HasCeoMD,HasChairman,HasPromoterOnBoard,HasDualityChairmanMD,IsFamilyManaged,HasFamilyChairman,HasFamilyChairmanAndCEO,CountFirstTermDir,CountFirstTermIndepDir,PercentFirstTermIndepDir,CountTurnOverIndepDir,HasIndepTurnOver,AvgTenureInYearsinCompTotal,AvgTenureInYearsinCompIndep,PercentRookieIndep,PercentNonRookieIndep,PercentRookieNonIndep,PercentNonRookieNonIndep,PercentRookie,PercentNonRookie,IsRookieBoard,JustOneRookie,TwoOrMoreRookies,CountMBA,PercentMBA,CountIndepMBA,PercentIndepMBA,CountRookieIndepMBA,PercentRookieIndepMBA,CountPhD,PercentPhD,CountIndepPhD,PercentIndepPhD,CountRookieIndepPhD,PercentRookieIndepPhD,CountFinanceXP,PercentFinanceXP,CountIndepFinanceXP,PercentIndepFinanceXP,CountRookieIndepFinanceXP,PercentRookieIndepFinanceXP,CountTechXP,PercentTechXP,CountIndepTechXP,PercentIndepTechXP,CountRookieIndepTechXP,PercentRookieIndepTechXP,CountFinanceSkill,PercentFinanceSkill,CountIndepFinanceSkill,PercentIndepFinanceSkill,CountRookieIndepFinanceSkill,PercentRookieIndepFinanceSkill,CountTechSkill,PercentTechSkill,CountIndepTechSkill,PercentIndepTechSkill,CountRookieIndepTechSkill,PercentRookieIndepTechSkill,CountRelatedIndustryXP,PercentRelatedIndustryXP,CountIndepRelatedIndustryXP,PercentIndepRelatedIndustryXP,CountRookieIndepRelatedIndustryXP,PercentRookieIndepRelatedIndustryXP,CountExecXP,PercentExecXP,CountIndepExecXP,PercentIndepExecXP,CountRookieIndepExecXP,PercentRookieIndepExecXP,CountPublicExecXP,PercentPublicExecXP,CountIndepPublicExecXP,PercentIndepPublicExecXP,CountRookieIndepPublicExecXP,PercentRookieIndepPublicExecXP,CountPrivateExecXP,PercentPrivateExecXP,CountIndepPrivateExecXP,PercentIndepPrivateExecXP,CountRookieIndepPrivateExecXP,PercentRookieIndepPrivateExecXP,CountCeoMDChairXP,PercentCeoMDChairXP,CountIndepCeoMDChairXP,PercentIndepCeoMDChairXP,CountRookieIndepCeoMDChairXP,PercentRookieIndepCeoMDChairXP,CountRookieAppoint,CountRookieIndepAppoint,CountNonRookieAppoint,CountNonRookieIndepAppoint,RookieAppointDummy,RookieIndepAppointDummy,NonRookieAppointDummy,NonRookieIndepAppointDummy,TotalDirCess,TotalRookieIndepDirCess,TotalDirAppoint,TotalRookieIndepDirAppoint,UniqueSkills,CountUniqueSkills,CountRetiringTermLimit,PercentRetiringTermLimit,TermLimitRetireesPcodeList,CountDirTurnOver_exits_retire,PercentDirTurnOver_exits_retire,CountDirTurnOver_real_exits,PercentDirTurnOver_real_exits,CountIndepDirTurnOver_exits_retire,PercentIndepDirTurnOver_exits_retire,CountIndepDirTurnOver_real_exits,PercentIndepDirTurnOver_real_exits,ZeroYearPCodeList,FirstYearPCodeList,TwoYearPCodeList,ThreeYearPCodeList,PCodeList,ZeroYearIndepPCodeList,FirstYearIndepPCodeList,TwoYearIndepPCodeList,ThreeYearIndepPCodeList,IndepPCodeList,BoardAbsenceMean,Academic_mean,Company Business_mean,combined_sustainability_mean,combined_entrepreneurial_mean,combined_compensation_mean,combined_conglomerate_experience_mean,combined_hr_mean,combined_technology_mean,combined_finance_accounting_mean,combined_governance_mean,combined_government_policy_mean,combined_international_mean,combined_leadership_mean,combined_legal_mean,combined_marketing_mean,combined_risk_management_mean,combined_scientific_mean,combined_strategic_planning_mean,combined_manufacturing_supply_chain_mean,combined_leadership_outside_board_mean,skill_committee_sustainability_mean,skill_committee_entrepreneurial_mean,skill_committee_compensation_mean,skill_committee_conglomerate_experience_mean,skill_committee_hr_mean,skill_committee_technology_mean,skill_committee_finance_accounting_mean,skill_committee_governance_mean,skill_committee_government_policy_mean,skill_committee_international_mean,skill_committee_leadership_mean,skill_committee_legal_mean,s

## Location Data

In [97]:
loc = pd.read_excel(rf"{supporting_folder_path}\Prowess IQ - Location Data.xlsx", skiprows = 5)
loc = loc.drop( ["Registrar Address", "Company Name"], axis = 1).dropna(how = "all")
loc

,Corporate Office,Head Office,Registered Office,Prowess company code
0,Mumbai,Vadodara,Vadodara,11
1,Mumbai,NaN,Mumbai,384392
2,Mumbai,NaN,Navi Mumbai,96387
3,Bengaluru,NaN,Bengaluru,36277
4,NaN,Mumbai,Pune,183396
...,...,...,...,...
2542,Surat,NaN,Surat,275663
2543,Bengaluru,NaN,Goa,397515
2544,Gurugram,NaN,Goa,275679
2545,Ahmedabad,Ahmedabad,Ahmedabad,41246


In [98]:
loc["Office"] = np.where( pd.isnull(loc["Head Office"]), loc["Corporate Office"], loc["Head Office"])
loc["Office"] = np.where( pd.isnull(loc["Office"]), loc["Registered Office"], loc["Office"])
loc2 = loc.drop( ["Corporate Office", "Head Office", "Registered Office"], axis = 1).dropna(how = "all")
loc2

,Prowess company code,Office
0,11,Vadodara
1,384392,Mumbai
2,96387,Mumbai
3,36277,Bengaluru
4,183396,Mumbai
...,...,...
2542,275663,Surat
2543,397515,Bengaluru
2544,275679,Gurugram
2545,41246,Ahmedabad


In [99]:
firmLoc = merged2.merge(loc2, on = ["Prowess company code"], how = "left")

In [100]:
firmLoc

,FirmID,Symbol,AsOnDate,TotDirCount,AsOnYear,LnBoardSize,FemaleDirCount,PercentWomenDir,Women_dummy,AvgAge,CountRookie,Rookie_dummy,CountNonRookie,CountIndep,PercentIndep,CountNonIndep,CountRookieIndep,CountRookieNonIndep,CountNonRookieIndep,CountNonRookieNonIndep,CountPromoterDir,CountBusyDir,PercentBusyDir,Busy_dummy,HasCeoMD,HasChairman,HasPromoterOnBoard,HasDualityChairmanMD,IsFamilyManaged,HasFamilyChairman,HasFamilyChairmanAndCEO,CountFirstTermDir,CountFirstTermIndepDir,PercentFirstTermIndepDir,CountTurnOverIndepDir,HasIndepTurnOver,AvgTenureInYearsinCompTotal,AvgTenureInYearsinCompIndep,PercentRookieIndep,PercentNonRookieIndep,PercentRookieNonIndep,PercentNonRookieNonIndep,PercentRookie,PercentNonRookie,IsRookieBoard,JustOneRookie,TwoOrMoreRookies,CountMBA,PercentMBA,CountIndepMBA,PercentIndepMBA,CountRookieIndepMBA,PercentRookieIndepMBA,CountPhD,PercentPhD,CountIndepPhD,PercentIndepPhD,CountRookieIndepPhD,PercentRookieIndepPhD,CountFinanceXP,PercentFinanceXP,CountIndepFinanceXP,PercentIndepFinanceXP,CountRookieIndepFinanceXP,PercentRookieIndepFinanceXP,CountTechXP,PercentTechXP,CountIndepTechXP,PercentIndepTechXP,CountRookieIndepTechXP,PercentRookieIndepTechXP,CountFinanceSkill,PercentFinanceSkill,CountIndepFinanceSkill,PercentIndepFinanceSkill,CountRookieIndepFinanceSkill,PercentRookieIndepFinanceSkill,CountTechSkill,PercentTechSkill,CountIndepTechSkill,PercentIndepTechSkill,CountRookieIndepTechSkill,PercentRookieIndepTechSkill,CountRelatedIndustryXP,PercentRelatedIndustryXP,CountIndepRelatedIndustryXP,PercentIndepRelatedIndustryXP,CountRookieIndepRelatedIndustryXP,PercentRookieIndepRelatedIndustryXP,CountExecXP,PercentExecXP,CountIndepExecXP,PercentIndepExecXP,CountRookieIndepExecXP,PercentRookieIndepExecXP,CountPublicExecXP,PercentPublicExecXP,CountIndepPublicExecXP,PercentIndepPublicExecXP,CountRookieIndepPublicExecXP,PercentRookieIndepPublicExecXP,CountPrivateExecXP,PercentPrivateExecXP,CountIndepPrivateExecXP,PercentIndepPrivateExecXP,CountRookieIndepPrivateExecXP,PercentRookieIndepPrivateExecXP,CountCeoMDChairXP,PercentCeoMDChairXP,CountIndepCeoMDChairXP,PercentIndepCeoMDChairXP,CountRookieIndepCeoMDChairXP,PercentRookieIndepCeoMDChairXP,CountRookieAppoint,CountRookieIndepAppoint,CountNonRookieAppoint,CountNonRookieIndepAppoint,RookieAppointDummy,RookieIndepAppointDummy,NonRookieAppointDummy,NonRookieIndepAppointDummy,TotalDirCess,TotalRookieIndepDirCess,TotalDirAppoint,TotalRookieIndepDirAppoint,UniqueSkills,CountUniqueSkills,CountRetiringTermLimit,PercentRetiringTermLimit,TermLimitRetireesPcodeList,CountDirTurnOver_exits_retire,PercentDirTurnOver_exits_retire,CountDirTurnOver_real_exits,PercentDirTurnOver_real_exits,CountIndepDirTurnOver_exits_retire,PercentIndepDirTurnOver_exits_retire,CountIndepDirTurnOver_real_exits,PercentIndepDirTurnOver_real_exits,ZeroYearPCodeList,FirstYearPCodeList,TwoYearPCodeList,ThreeYearPCodeList,PCodeList,ZeroYearIndepPCodeList,FirstYearIndepPCodeList,TwoYearIndepPCodeList,ThreeYearIndepPCodeList,IndepPCodeList,BoardAbsenceMean,Academic_mean,Company Business_mean,combined_sustainability_mean,combined_entrepreneurial_mean,combined_compensation_mean,combined_conglomerate_experience_mean,combined_hr_mean,combined_technology_mean,combined_finance_accounting_mean,combined_governance_mean,combined_government_policy_mean,combined_international_mean,combined_leadership_mean,combined_legal_mean,combined_marketing_mean,combined_risk_management_mean,combined_scientific_mean,combined_strategic_planning_mean,combined_manufacturing_supply_chain_mean,combined_leadership_outside_board_mean,skill_committee_sustainability_mean,skill_committee_entrepreneurial_mean,skill_committee_compensation_mean,skill_committee_conglomerate_experience_mean,skill_committee_hr_mean,skill_committee_technology_mean,skill_committee_finance_accounting_mean,skill_committee_governance_mean,skill_committee_government_policy_mean,skill_committee_international_mean,skill_committee_leadership_mean,skill_committee_legal_mean,s

# Further Calc

In [101]:
firm0 = firmLoc.copy()

firm = firm0.rename({"Paid up preference capital (net of forfeited preference capital) (Old Sch. VI)":"Paid up preference capital_net of forfeited preference capital_old",
                     "Paid up preference capital (net of forfeited preference capital)":"Paid up preference capital_net of forfeited preference capital",
                     "Indian Promoters (In %) - Shares held":"Indian Promoters_percent",
                     "Total Shares (In %) - Shares held":"Total Shares_percent_Shares held",
                     "Foreign Promoters (In %) - Shares held":"Foreign Promoters_percent",
                     "Non-promoters (In %) - Shares held":"Nonpromoters_percent",
                     "Non-promoter Institutions (In %) - Shares held":"NonpromoterInstitutions_percent",
                     "Non-promoter Non-institutions (In %) - Shares held":"Nonpromoter Noninstitutions_percent",
                     "Trade payables (Old Sch. VI)":"Trade payables_old",
                     "Debt to equity ratio (times)":"Debt to equity ratio",
                     "Cash (outflow) due to purchase of fixed assets":"Cash outflow due to purchase of fixed assets",
                     "Net cash inflow or (outflow) from investing activities":"Net cash inflow or outflow from investing activities",
                     "Cash (outflow) due to purchase of Intangible assets":"Cash outflow due to purchase of Intangible assets",
                     "Research & development expenses":"Research development expenses",
                     "Inter-corporate loans (Old Sch. VI)":"Intercorporate loans_old",
                     "Promoters (In %) - Shares held":"Promoters_percent",
                     "Inter-corporate loans & borrowings (as per nbfc norms)":"Inter-corporate loans & borrowings_nbfc norms",
                     "Inter-corporate loans (Old Sch. VI)":"Inter-corporate loans_old sch",
                     "Secured inter-corporate loans (Old Sch. VI)":"Secured inter-corporate loans_old sch",
                     "Unsecured inter-corporate loans (Old Sch. VI)":"Unsecured inter-corporate loans_old sch"
                    }, axis = 1)

firm

,FirmID,Symbol,AsOnDate,TotDirCount,AsOnYear,LnBoardSize,FemaleDirCount,PercentWomenDir,Women_dummy,AvgAge,CountRookie,Rookie_dummy,CountNonRookie,CountIndep,PercentIndep,CountNonIndep,CountRookieIndep,CountRookieNonIndep,CountNonRookieIndep,CountNonRookieNonIndep,CountPromoterDir,CountBusyDir,PercentBusyDir,Busy_dummy,HasCeoMD,HasChairman,HasPromoterOnBoard,HasDualityChairmanMD,IsFamilyManaged,HasFamilyChairman,HasFamilyChairmanAndCEO,CountFirstTermDir,CountFirstTermIndepDir,PercentFirstTermIndepDir,CountTurnOverIndepDir,HasIndepTurnOver,AvgTenureInYearsinCompTotal,AvgTenureInYearsinCompIndep,PercentRookieIndep,PercentNonRookieIndep,PercentRookieNonIndep,PercentNonRookieNonIndep,PercentRookie,PercentNonRookie,IsRookieBoard,JustOneRookie,TwoOrMoreRookies,CountMBA,PercentMBA,CountIndepMBA,PercentIndepMBA,CountRookieIndepMBA,PercentRookieIndepMBA,CountPhD,PercentPhD,CountIndepPhD,PercentIndepPhD,CountRookieIndepPhD,PercentRookieIndepPhD,CountFinanceXP,PercentFinanceXP,CountIndepFinanceXP,PercentIndepFinanceXP,CountRookieIndepFinanceXP,PercentRookieIndepFinanceXP,CountTechXP,PercentTechXP,CountIndepTechXP,PercentIndepTechXP,CountRookieIndepTechXP,PercentRookieIndepTechXP,CountFinanceSkill,PercentFinanceSkill,CountIndepFinanceSkill,PercentIndepFinanceSkill,CountRookieIndepFinanceSkill,PercentRookieIndepFinanceSkill,CountTechSkill,PercentTechSkill,CountIndepTechSkill,PercentIndepTechSkill,CountRookieIndepTechSkill,PercentRookieIndepTechSkill,CountRelatedIndustryXP,PercentRelatedIndustryXP,CountIndepRelatedIndustryXP,PercentIndepRelatedIndustryXP,CountRookieIndepRelatedIndustryXP,PercentRookieIndepRelatedIndustryXP,CountExecXP,PercentExecXP,CountIndepExecXP,PercentIndepExecXP,CountRookieIndepExecXP,PercentRookieIndepExecXP,CountPublicExecXP,PercentPublicExecXP,CountIndepPublicExecXP,PercentIndepPublicExecXP,CountRookieIndepPublicExecXP,PercentRookieIndepPublicExecXP,CountPrivateExecXP,PercentPrivateExecXP,CountIndepPrivateExecXP,PercentIndepPrivateExecXP,CountRookieIndepPrivateExecXP,PercentRookieIndepPrivateExecXP,CountCeoMDChairXP,PercentCeoMDChairXP,CountIndepCeoMDChairXP,PercentIndepCeoMDChairXP,CountRookieIndepCeoMDChairXP,PercentRookieIndepCeoMDChairXP,CountRookieAppoint,CountRookieIndepAppoint,CountNonRookieAppoint,CountNonRookieIndepAppoint,RookieAppointDummy,RookieIndepAppointDummy,NonRookieAppointDummy,NonRookieIndepAppointDummy,TotalDirCess,TotalRookieIndepDirCess,TotalDirAppoint,TotalRookieIndepDirAppoint,UniqueSkills,CountUniqueSkills,CountRetiringTermLimit,PercentRetiringTermLimit,TermLimitRetireesPcodeList,CountDirTurnOver_exits_retire,PercentDirTurnOver_exits_retire,CountDirTurnOver_real_exits,PercentDirTurnOver_real_exits,CountIndepDirTurnOver_exits_retire,PercentIndepDirTurnOver_exits_retire,CountIndepDirTurnOver_real_exits,PercentIndepDirTurnOver_real_exits,ZeroYearPCodeList,FirstYearPCodeList,TwoYearPCodeList,ThreeYearPCodeList,PCodeList,ZeroYearIndepPCodeList,FirstYearIndepPCodeList,TwoYearIndepPCodeList,ThreeYearIndepPCodeList,IndepPCodeList,BoardAbsenceMean,Academic_mean,Company Business_mean,combined_sustainability_mean,combined_entrepreneurial_mean,combined_compensation_mean,combined_conglomerate_experience_mean,combined_hr_mean,combined_technology_mean,combined_finance_accounting_mean,combined_governance_mean,combined_government_policy_mean,combined_international_mean,combined_leadership_mean,combined_legal_mean,combined_marketing_mean,combined_risk_management_mean,combined_scientific_mean,combined_strategic_planning_mean,combined_manufacturing_supply_chain_mean,combined_leadership_outside_board_mean,skill_committee_sustainability_mean,skill_committee_entrepreneurial_mean,skill_committee_compensation_mean,skill_committee_conglomerate_experience_mean,skill_committee_hr_mean,skill_committee_technology_mean,skill_committee_finance_accounting_mean,skill_committee_governance_mean,skill_committee_government_policy_mean,skill_committee_international_mean,skill_committee_leadership_mean,skill_committee_legal_mean,s

## Calculations

In [102]:
firm["NoFamilyControl"] = np.where( (firm["IsFamilyManaged"] == 0) & (firm["HasFamilyChairman"] == 0)
                                   & (firm["HasFamilyChairmanAndCEO"] == 0), 1, 0)

firm["PBOA_calc"] = firm["PBDITA"].fillna(0) / firm["Total assets"]
firm["ROA_calc"] = firm["Profit after tax"].fillna(0) / firm["Total assets"]
firm["ln_totalassets"] = np.log(firm["Total assets"])

firm["Book value of common stock"] = firm["BVPS"].fillna(0) * firm["Shares Outstanding "].fillna(0)

firm["TobinQ_Debt1"] = (firm["Debt"].fillna(0) + firm["Paid up preference capital_net of forfeited preference capital"].fillna(0)
                        + firm["Market Capitalisation"].fillna(0)) / firm["Total assets"]
firm["ln_TobinQ_Debt1"] = np.log(firm["TobinQ_Debt1"])

firm["TobinQ_longborrowincl2"] = (firm["Long term borrowings incl current portion"].fillna(0)
                                  + firm["Paid up preference capital_net of forfeited preference capital"].fillna(0)
                                  + firm["Market Capitalisation"].fillna(0)) / firm["Total assets"]

firm["TobinQ_longborrowexcl3"] = (firm["Long term borrowings excl current portion"].fillna(0)
                                  + firm["Paid up preference capital_net of forfeited preference capital"].fillna(0)
                                  + firm["Market Capitalisation"].fillna(0)) / firm["Total assets"]

firm["ln_TobinQ_longborrowincl2"] = np.log(firm["TobinQ_longborrowincl2"])
firm["ln_TobinQ_longborrowexcl3"] = np.log(firm["TobinQ_longborrowexcl3"])


firm["TobinQ_BVPS"] = (firm["Total assets"].fillna(0)
                       + firm["Market Capitalisation"].fillna(0)
                       - firm["Book value of common stock"].fillna(0)) / firm["Total assets"]

firm["BookLeverage"] = (firm["Long term borrowings incl current portion"].fillna(0)
                        + firm["Trade payables_old"].fillna(0)) / firm["Total assets"]

firm["Debt_TotalAssets"] = firm["Debt"].fillna(0) / firm["Total assets"]
firm["RD to Assets"] = firm["Research development expenses"].fillna(0) / firm["Total assets"]
firm["ln_rdtoassets"] = np.log(firm["RD to Assets"])

firm["ln_marcap"] = np.log(firm["Market Capitalisation"])

firm["FirmAge"] = firm["AsOnYear"].fillna(0) - firm["Incorporation year"]
firm["lnage"] = np.log(firm["FirmAge"])

firm["govtdummy"] = np.where( firm["Ownership group code"] <= 20000000000, 1, 0)

fincode = [649, 64191, 64192, 64300, 64920, 64990, 65110, 65120, 66190, 66301]
firm["findummy"] = np.where( firm["NIC code"].isin(fincode), 1, 0)

firm["promoterholding20"] = np.where( firm["Promoters_percent"] >= 20,1,0)
firm["promoterholding25"] = np.where( firm["Promoters_percent"] >= 25,1,0)
firm["promoterholding30"] = np.where( firm["Promoters_percent"] >= 30,1,0)
firm["promoterholding40"] = np.where( firm["Promoters_percent"] >= 40,1,0)
firm["promoterholding50"] = np.where( firm["Promoters_percent"] >= 50,1,0)

firm["FamilyOwnedFamilyCeoChair20"] = np.where( (firm["HasFamilyChairmanAndCEO"] == 1) & (firm["promoterholding20"] == 1),1,0)
firm["FamilyOwnedFamilyCeoChair25"] = np.where( (firm["HasFamilyChairmanAndCEO"] == 1) & (firm["promoterholding25"] == 1),1,0)
firm["FamilyOwnedFamilyCeoChair30"] = np.where( (firm["HasFamilyChairmanAndCEO"] == 1) & (firm["promoterholding30"] == 1),1,0)
firm["FamilyOwnedFamilyCeoChair40"] = np.where( (firm["HasFamilyChairmanAndCEO"] == 1) & (firm["promoterholding40"] == 1),1,0)
firm["FamilyOwnedFamilyCeoChair50"] = np.where( (firm["HasFamilyChairmanAndCEO"] == 1) & (firm["promoterholding50"] == 1),1,0)

firm["FamilyOwnedFamilyChair20"] = np.where( (firm["HasFamilyChairman"] == 1) & (firm["promoterholding20"] == 1),1,0)
firm["FamilyOwnedFamilyChair25"] = np.where( (firm["HasFamilyChairman"] == 1) & (firm["promoterholding25"] == 1),1,0)
firm["FamilyOwnedFamilyChair30"] = np.where( (firm["HasFamilyChairman"] == 1) & (firm["promoterholding30"] == 1),1,0)
firm["FamilyOwnedFamilyChair40"] = np.where( (firm["HasFamilyChairman"] == 1) & (firm["promoterholding40"] == 1),1,0)
firm["FamilyOwnedFamilyChair50"] = np.where( (firm["HasFamilyChairman"] == 1) & (firm["promoterholding50"] == 1),1,0)

firm["FamilyOwnedFamilyManaged20"] = np.where( (firm["IsFamilyManaged"] == 1) & (firm["promoterholding20"] == 1),1,0)
firm["FamilyOwnedFamilyManaged25"] = np.where( (firm["IsFamilyManaged"] == 1) & (firm["promoterholding25"] == 1),1,0)
firm["FamilyOwnedFamilyManaged30"] = np.where( (firm["IsFamilyManaged"] == 1) & (firm["promoterholding30"] == 1),1,0)
firm["FamilyOwnedFamilyManaged40"] = np.where( (firm["IsFamilyManaged"] == 1) & (firm["promoterholding40"] == 1),1,0)
firm["FamilyOwnedFamilyManaged50"] = np.where( (firm["IsFamilyManaged"] == 1) & (firm["promoterholding50"] == 1),1,0)


C:\Users\SHIVAM\anaconda3\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\SHIVAM\anaconda3\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\SHIVAM\anaconda3\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\SHIVAM\anaconda3\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\SHIVAM\anaconda3\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\SHIVAM\anaconda3\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountere

In [103]:
firm.columns.to_list()

['FirmID',
 'Symbol',
 'AsOnDate',
 'TotDirCount',
 'AsOnYear',
 'LnBoardSize',
 'FemaleDirCount',
 'PercentWomenDir',
 'Women_dummy',
 'AvgAge',
 'CountRookie',
 'Rookie_dummy',
 'CountNonRookie',
 'CountIndep',
 'PercentIndep',
 'CountNonIndep',
 'CountRookieIndep',
 'CountRookieNonIndep',
 'CountNonRookieIndep',
 'CountNonRookieNonIndep',
 'CountPromoterDir',
 'CountBusyDir',
 'PercentBusyDir',
 'Busy_dummy',
 'HasCeoMD',
 'HasChairman',
 'HasPromoterOnBoard',
 'HasDualityChairmanMD',
 'IsFamilyManaged',
 'HasFamilyChairman',
 'HasFamilyChairmanAndCEO',
 'CountFirstTermDir',
 'CountFirstTermIndepDir',
 'PercentFirstTermIndepDir',
 'CountTurnOverIndepDir',
 'HasIndepTurnOver',
 'AvgTenureInYearsinCompTotal',
 'AvgTenureInYearsinCompIndep',
 'PercentRookieIndep',
 'PercentNonRookieIndep',
 'PercentRookieNonIndep',
 'PercentNonRookieNonIndep',
 'PercentRookie',
 'PercentNonRookie',
 'IsRookieBoard',
 'JustOneRookie',
 'TwoOrMoreRookies',
 'CountMBA',
 'PercentMBA',
 'CountIndepMBA',
 '

In [104]:
firm = firm.replace([np.inf, -np.inf], np.nan)
firm.describe()

,FirmID,AsOnDate,TotDirCount,AsOnYear,LnBoardSize,FemaleDirCount,PercentWomenDir,Women_dummy,CountRookie,Rookie_dummy,CountNonRookie,CountIndep,PercentIndep,CountNonIndep,CountRookieIndep,CountRookieNonIndep,CountNonRookieIndep,CountNonRookieNonIndep,CountPromoterDir,CountBusyDir,PercentBusyDir,Busy_dummy,CountFirstTermDir,CountFirstTermIndepDir,PercentFirstTermIndepDir,CountTurnOverIndepDir,HasIndepTurnOver,AvgTenureInYearsinCompTotal,AvgTenureInYearsinCompIndep,PercentRookieIndep,PercentNonRookieIndep,PercentRookieNonIndep,PercentNonRookieNonIndep,PercentRookie,PercentNonRookie,IsRookieBoard,JustOneRookie,TwoOrMoreRookies,CountMBA,PercentMBA,CountIndepMBA,PercentIndepMBA,CountRookieIndepMBA,PercentRookieIndepMBA,CountPhD,PercentPhD,CountIndepPhD,PercentIndepPhD,CountRookieIndepPhD,PercentRookieIndepPhD,CountFinanceXP,PercentFinanceXP,CountIndepFinanceXP,PercentIndepFinanceXP,CountRookieIndepFinanceXP,PercentRookieIndepFinanceXP,CountTechXP,PercentTechXP,CountIndepTechXP,PercentIndepTechXP,CountRookieIndepTechXP,PercentRookieIndepTechXP,CountFinanceSkill,PercentFinanceSkill,CountIndepFinanceSkill,PercentIndepFinanceSkill,CountRookieIndepFinanceSkill,PercentRookieIndepFinanceSkill,CountTechSkill,PercentTechSkill,CountIndepTechSkill,PercentIndepTechSkill,CountRookieIndepTechSkill,PercentRookieIndepTechSkill,CountRelatedIndustryXP,PercentRelatedIndustryXP,CountIndepRelatedIndustryXP,PercentIndepRelatedIndustryXP,CountRookieIndepRelatedIndustryXP,PercentRookieIndepRelatedIndustryXP,CountExecXP,PercentExecXP,CountIndepExecXP,PercentIndepExecXP,CountRookieIndepExecXP,PercentRookieIndepExecXP,CountPublicExecXP,PercentPublicExecXP,CountIndepPublicExecXP,PercentIndepPublicExecXP,CountRookieIndepPublicExecXP,PercentRookieIndepPublicExecXP,CountPrivateExecXP,PercentPrivateExecXP,CountIndepPrivateExecXP,PercentIndepPrivateExecXP,CountRookieIndepPrivateExecXP,PercentRookieIndepPrivateExecXP,CountCeoMDChairXP,PercentCeoMDChairXP,CountIndepCeoMDChairXP,PercentIndepCeoMDChairXP,CountRookieIndepCeoMDChairXP,PercentRookieIndepCeoMDChairXP,CountRookieAppoint,CountRookieIndepAppoint,CountNonRookieAppoint,CountNonRookieIndepAppoint,RookieAppointDummy,RookieIndepAppointDummy,NonRookieAppointDummy,NonRookieIndepAppointDummy,TotalDirCess,TotalRookieIndepDirCess,TotalDirAppoint,TotalRookieIndepDirAppoint,CountUniqueSkills,CountRetiringTermLimit,PercentRetiringTermLimit,CountDirTurnOver_exits_retire,PercentDirTurnOver_exits_retire,CountDirTurnOver_real_exits,PercentDirTurnOver_real_exits,CountIndepDirTurnOver_exits_retire,PercentIndepDirTurnOver_exits_retire,CountIndepDirTurnOver_real_exits,PercentIndepDirTurnOver_real_exits,BoardAbsenceMean,Academic_mean,Company Business_mean,combined_sustainability_mean,combined_entrepreneurial_mean,combined_compensation_mean,combined_conglomerate_experience_mean,combined_hr_mean,combined_technology_mean,combined_finance_accounting_mean,combined_governance_mean,combined_government_policy_mean,combined_international_mean,combined_leadership_mean,combined_legal_mean,combined_marketing_mean,combined_risk_management_mean,combined_scientific_mean,combined_strategic_planning_mean,combined_manufacturing_supply_chain_mean,combined_leadership_outside_board_mean,skill_committee_sustainability_mean,skill_committee_entrepreneurial_mean,skill_committee_compensation_mean,skill_committee_conglomerate_experience_mean,skill_committee_hr_mean,skill_committee_technology_mean,skill_committee_finance_accounting_mean,skill_committee_governance_mean,skill_committee_government_policy_mean,skill_committee_international_mean,skill_committee_leadership_mean,skill_committee_legal_mean,skill_committee_marketing_mean,skill_committee_risk_management_mean,skill_committee_scientific_mean,skill_committee_strategic_planning_mean,skill_committee_manufacturing_supply_chain_mean,skill_committee_leadership_outside_board_mean,board_consumer discretionary_mean,board_research & development_mean,board_legal/compliance_mean,board_utilities_mean,board_healthca

In [105]:
firm.loc[ firm["AvgTenureInYearsinCompTotal"] < 0]

,FirmID,Symbol,AsOnDate,TotDirCount,AsOnYear,LnBoardSize,FemaleDirCount,PercentWomenDir,Women_dummy,AvgAge,CountRookie,Rookie_dummy,CountNonRookie,CountIndep,PercentIndep,CountNonIndep,CountRookieIndep,CountRookieNonIndep,CountNonRookieIndep,CountNonRookieNonIndep,CountPromoterDir,CountBusyDir,PercentBusyDir,Busy_dummy,HasCeoMD,HasChairman,HasPromoterOnBoard,HasDualityChairmanMD,IsFamilyManaged,HasFamilyChairman,HasFamilyChairmanAndCEO,CountFirstTermDir,CountFirstTermIndepDir,PercentFirstTermIndepDir,CountTurnOverIndepDir,HasIndepTurnOver,AvgTenureInYearsinCompTotal,AvgTenureInYearsinCompIndep,PercentRookieIndep,PercentNonRookieIndep,PercentRookieNonIndep,PercentNonRookieNonIndep,PercentRookie,PercentNonRookie,IsRookieBoard,JustOneRookie,TwoOrMoreRookies,CountMBA,PercentMBA,CountIndepMBA,PercentIndepMBA,CountRookieIndepMBA,PercentRookieIndepMBA,CountPhD,PercentPhD,CountIndepPhD,PercentIndepPhD,CountRookieIndepPhD,PercentRookieIndepPhD,CountFinanceXP,PercentFinanceXP,CountIndepFinanceXP,PercentIndepFinanceXP,CountRookieIndepFinanceXP,PercentRookieIndepFinanceXP,CountTechXP,PercentTechXP,CountIndepTechXP,PercentIndepTechXP,CountRookieIndepTechXP,PercentRookieIndepTechXP,CountFinanceSkill,PercentFinanceSkill,CountIndepFinanceSkill,PercentIndepFinanceSkill,CountRookieIndepFinanceSkill,PercentRookieIndepFinanceSkill,CountTechSkill,PercentTechSkill,CountIndepTechSkill,PercentIndepTechSkill,CountRookieIndepTechSkill,PercentRookieIndepTechSkill,CountRelatedIndustryXP,PercentRelatedIndustryXP,CountIndepRelatedIndustryXP,PercentIndepRelatedIndustryXP,CountRookieIndepRelatedIndustryXP,PercentRookieIndepRelatedIndustryXP,CountExecXP,PercentExecXP,CountIndepExecXP,PercentIndepExecXP,CountRookieIndepExecXP,PercentRookieIndepExecXP,CountPublicExecXP,PercentPublicExecXP,CountIndepPublicExecXP,PercentIndepPublicExecXP,CountRookieIndepPublicExecXP,PercentRookieIndepPublicExecXP,CountPrivateExecXP,PercentPrivateExecXP,CountIndepPrivateExecXP,PercentIndepPrivateExecXP,CountRookieIndepPrivateExecXP,PercentRookieIndepPrivateExecXP,CountCeoMDChairXP,PercentCeoMDChairXP,CountIndepCeoMDChairXP,PercentIndepCeoMDChairXP,CountRookieIndepCeoMDChairXP,PercentRookieIndepCeoMDChairXP,CountRookieAppoint,CountRookieIndepAppoint,CountNonRookieAppoint,CountNonRookieIndepAppoint,RookieAppointDummy,RookieIndepAppointDummy,NonRookieAppointDummy,NonRookieIndepAppointDummy,TotalDirCess,TotalRookieIndepDirCess,TotalDirAppoint,TotalRookieIndepDirAppoint,UniqueSkills,CountUniqueSkills,CountRetiringTermLimit,PercentRetiringTermLimit,TermLimitRetireesPcodeList,CountDirTurnOver_exits_retire,PercentDirTurnOver_exits_retire,CountDirTurnOver_real_exits,PercentDirTurnOver_real_exits,CountIndepDirTurnOver_exits_retire,PercentIndepDirTurnOver_exits_retire,CountIndepDirTurnOver_real_exits,PercentIndepDirTurnOver_real_exits,ZeroYearPCodeList,FirstYearPCodeList,TwoYearPCodeList,ThreeYearPCodeList,PCodeList,ZeroYearIndepPCodeList,FirstYearIndepPCodeList,TwoYearIndepPCodeList,ThreeYearIndepPCodeList,IndepPCodeList,BoardAbsenceMean,Academic_mean,Company Business_mean,combined_sustainability_mean,combined_entrepreneurial_mean,combined_compensation_mean,combined_conglomerate_experience_mean,combined_hr_mean,combined_technology_mean,combined_finance_accounting_mean,combined_governance_mean,combined_government_policy_mean,combined_international_mean,combined_leadership_mean,combined_legal_mean,combined_marketing_mean,combined_risk_management_mean,combined_scientific_mean,combined_strategic_planning_mean,combined_manufacturing_supply_chain_mean,combined_leadership_outside_board_mean,skill_committee_sustainability_mean,skill_committee_entrepreneurial_mean,skill_committee_compensation_mean,skill_committee_conglomerate_experience_mean,skill_committee_hr_mean,skill_committee_technology_mean,skill_committee_finance_accounting_mean,skill_committee_governance_mean,skill_committee_government_policy_mean,skill_committee_international_mean,skill_committee_leadership_mean,skill_committee_legal_mean,s

In [106]:
firm.to_pickle(rf"{output_folder_path}\Main_Firm_No Loc_Fin_Calc.pkl")
firm.to_csv(rf"{output_folder_path}\Main_Firm_No Loc_Fin_Calc v210625.csv")